In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

# Detect project root
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Project directories
CORRUPTED_DIR = PROJECT_ROOT / "data" / "corrupted"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
RESULTS_DIR = PROJECT_ROOT / "results" / "cleaning"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Experiment configuration
SEEDS = [42, 123, 2026]
CORRUPTION_RATE = 0.10

print("ENVIRONMENT CHECK:")
print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\nPATH CHECK:")
print("Corrupted directory exists:", CORRUPTED_DIR.exists())
print("Clean directory exists:", CLEAN_DIR.exists())
print("Results directory exists:", RESULTS_DIR.exists())


ENVIRONMENT CHECK:
Python: C:\Users\ayman\OneDrive\Desktop\hitl-tabular-data-preparation\.venv\Scripts\python.exe
Project root: C:\Users\ayman\OneDrive\Desktop\hitl-tabular-data-preparation
Pandas: 3.1.1
NumPy: 2.5.5
Scikit-learn: 1.9.1

PATH CHECK:
Corrupted directory exists: True
Clean directory exists: True
Results directory exists: True


In [2]:

# Load corrupted training data (Seed 42)

seed = 42

corrupted_file = CORRUPTED_DIR / f"seed_{seed}" / "corrupted_train.csv"

df_corrupted = pd.read_csv(corrupted_file)

print("DATASET OVERVIEW:")
print("Dataset shape:", df_corrupted.shape)
print("\nColumn names")
print(df_corrupted.columns.tolist())

print("\nMISSING VALUES:")
missing_summary = df_corrupted.isnull().sum()
print(missing_summary[missing_summary > 0])

print("\nTotal missing cells:", df_corrupted.isnull().sum().sum())

print("\nDATA TYPES")
print(df_corrupted.dtypes)

print("\nFIRST 5 ROWS:")
display(df_corrupted.head())


DATASET OVERVIEW:
Dataset shape: (4000, 16)

Column names
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

MISSING VALUES:
age           97
job          109
marital      100
education     79
default      109
balance       74
housing      117
loan         112
contact       92
day           99
month         97
campaign     112
pdays         99
previous      91
poutcome     113
dtype: int64

Total missing cells: 1500

DATA TYPES
age          float64
job              str
marital          str
education        str
default          str
balance      float64
housing          str
loan             str
contact          str
day          float64
month            str
campaign     float64
pdays        float64
previous     float64
poutcome         str
y                str
dtype: object

FIRST 5 ROWS:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y
0,36.0,managemnt,single,tertiary,no,832.0,yes,NaN,cellular,-5.0,apr,2.0,-1.0,0.0,unknown,no
1,54.0,admin.,married,secondary,no,NaN,yes,no,cellular,6.0,may,2.0,343.0,2.0,failure,no
2,52.0,entrepreneur,married,primary,no,-104.0,no,NaN,cellular,10.0,jul,1.0,-1.0,0.0,NaN,no
3,44.0,management,divorced,tertiary,nox,-90.0,yes,no,cellular,21.0,jul,1.0,-1.0,0.0,unknown,no
4,38.0,housemaid,married,primary,NaN,196.0,NO,no,cellular,5.0,apr,1.0,292.0,0.0,unknown,yes


In [3]:

# Inspect categorical values

categorical_cols = df_corrupted.select_dtypes(
    include=["str", "object"]
).columns.tolist()

print("========== CATEGORICAL VALUES ==========")

for col in categorical_cols:
    print(f"\n{col}:")
    print(df_corrupted[col].value_counts(dropna=False).head(20))


# Inspect numerical statistics

numerical_cols = df_corrupted.select_dtypes(
    include=["number"]
).columns.tolist()

print("\n========== NUMERICAL STATISTICS ==========")

display(
    df_corrupted[numerical_cols].describe().T
)


========== CATEGORICAL VALUES ==========

job:
job
blue-collar      785
management       741
technician       560
admin.           424
services         323
retired          182
unemployed       126
housemaid        116
self-employed    115
NaN              109
entrepreneur     101
student           67
 MANAGEMENT       36
 BLUE-COLLAR      31
 TECHNICIAN       31
 SERVICES         20
unknown           18
 ADMIN.           15
 RETIRED          10
manageent          8
Name: count, dtype: int64

marital:
marital
married       2133
single         992
divorced       444
NaN            100
 MARRIED        85
 SINGLE         47
maried          33
 DIVORCED       26
marrid          18
mrried          17
arried          16
marrie          14
sigle           12
marred          12
sngle           11
singe           10
ingle            5
singl            5
ivorced          4
dvorced          4
Name: count, dtype: int64

education:
education
secondary      1831
tertiary       1052
primary         5

,count,mean,std,min,25%,50%,75%,max
age,3903.0,40.635665,13.220733,-14.0,32.00,39.0,49.0,109.0
balance,3926.0,1343.845644,3573.498123,-9528.0,50.25,443.0,1557.5,52527.0
day,3901.0,15.560369,10.545689,-24.0,8.00,16.0,22.0,55.0
campaign,3888.0,2.763117,3.959430,-9.0,1.00,2.0,3.0,73.0
pdays,3901.0,38.484491,120.555101,-294.0,-1.00,-1.0,-1.0,842.0
previous,3909.0,0.553594,5.994411,-14.0,0.00,0.0,0.0,275.0


In [4]:

# Define valid categorical values for Bank Marketing

VALID_CATEGORIES = {
    "job": [
        "admin.", "blue-collar", "entrepreneur", "housemaid",
        "management", "retired", "self-employed", "services",
        "student", "technician", "unemployed", "unknown"
    ],
    "marital": ["divorced", "married", "single"],
    "education": ["primary", "secondary", "tertiary", "unknown"],
    "default": ["yes", "no"],
    "housing": ["yes", "no"],
    "loan": ["yes", "no"],
    "contact": ["cellular", "telephone", "unknown"],
    "month": [
        "jan", "feb", "mar", "apr", "may", "jun",
        "jul", "aug", "sep", "oct", "nov", "dec"
    ],
    "poutcome": ["failure", "other", "success", "unknown"]
}

print("CANONICAL CATEGORY DICTIONARY:")

for column, values in VALID_CATEGORIES.items():
    print(f"{column}: {values}")

print("\nTotal categorical columns:", len(VALID_CATEGORIES))


CANONICAL CATEGORY DICTIONARY:
job: ['admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown']
marital: ['divorced', 'married', 'single']
education: ['primary', 'secondary', 'tertiary', 'unknown']
default: ['yes', 'no']
housing: ['yes', 'no']
loan: ['yes', 'no']
contact: ['cellular', 'telephone', 'unknown']
month: ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
poutcome: ['failure', 'other', 'success', 'unknown']

Total categorical columns: 9


In [5]:

from difflib import get_close_matches

# Numerical columns and domain constraints
NUMERICAL_COLS = [
    "age", "balance", "day", "campaign",
    "pdays", "previous"
]

# Conservative domain rules based on Bank Marketing fields
DOMAIN_RULES = {
    "age": lambda x: 18 <= x <= 100,
    "day": lambda x: 1 <= x <= 31,
    "campaign": lambda x: x >= 1,
    "pdays": lambda x: x >= -1,
    "previous": lambda x: x >= 0
}

def rule_based_clean(data):
    cleaned = data.copy()
    repair_records = []

    # Preserve original values for the repair log
    original_data = data.copy()


    # 1. Categorical cleaning

    for col, valid_values in VALID_CATEGORIES.items():

        for idx in cleaned.index:
            value = cleaned.at[idx, col]

            if pd.isna(value):
                continue

            original_value = value
            normalized = str(value).strip().lower()
            repaired = normalized
            operation = None

            # Normalize case and whitespace
            if normalized != str(value):
                operation = "format_normalization"

            # Correct spelling using canonical categories
            if repaired not in valid_values:
                matches = get_close_matches(
                    repaired,
                    valid_values,
                    n=1,
                    cutoff=0.80
                )

                if matches:
                    repaired = matches[0]
                    operation = "fuzzy_matching"

            if repaired != normalized or operation == "format_normalization":
                cleaned.at[idx, col] = repaired

                repair_records.append({
                    "row_index": idx,
                    "column": col,
                    "original_value": original_value,
                    "repaired_value": repaired,
                    "operation": operation
                })

        # Impute remaining missing categorical values
        mode_values = cleaned[col].dropna().mode()

        if not mode_values.empty:
            mode_value = mode_values.iloc[0]

            missing_indices = cleaned.index[cleaned[col].isna()]

            for idx in missing_indices:
                cleaned.at[idx, col] = mode_value

                repair_records.append({
                    "row_index": idx,
                    "column": col,
                    "original_value": np.nan,
                    "repaired_value": mode_value,
                    "operation": "categorical_mode_imputation"
                })

 
    # 2. Numerical cleaning
 
    for col in NUMERICAL_COLS:

        cleaned[col] = pd.to_numeric(
            cleaned[col], errors="coerce"
        )

        # Detect invalid values using domain rules
        if col in DOMAIN_RULES:
            invalid_mask = (
                cleaned[col].notna()
                & ~cleaned[col].apply(DOMAIN_RULES[col])
            )
        else:
            invalid_mask = pd.Series(
                False, index=cleaned.index
            )

        # Replace invalid values with missing values
        cleaned.loc[invalid_mask, col] = np.nan

        # Impute missing and invalid numerical values
        median_value = cleaned[col].median()

        if pd.notna(median_value):
            missing_indices = cleaned.index[cleaned[col].isna()]

            for idx in missing_indices:
                original_value = original_data.at[idx, col]

                if pd.isna(original_value):
                    operation = "numerical_median_imputation"
                else:
                    operation = "invalid_value_median_imputation"

                cleaned.at[idx, col] = median_value

                repair_records.append({
                    "row_index": idx,
                    "column": col,
                    "original_value": original_value,
                    "repaired_value": median_value,
                    "operation": operation
                })

    repair_log = pd.DataFrame(repair_records)

    return cleaned, repair_log


print("Rule-Based Cleaner function defined successfully.")


Rule-Based Cleaner function defined successfully.


In [6]:

# Apply Rule-Based Cleaning to Seed 42

df_rule_cleaned, repair_log = rule_based_clean(df_corrupted)

print("========== CLEANING RESULTS ==========")

print("Original shape:", df_corrupted.shape)
print("Cleaned shape:", df_rule_cleaned.shape)

print("\nRemaining missing values:",
      df_rule_cleaned.isnull().sum().sum())

print("\nTotal repair operations:", len(repair_log))

print("\n========== OPERATIONS DISTRIBUTION ==========")

display(
    repair_log["operation"]
    .value_counts()
    .rename_axis("operation")
    .to_frame("count")
)

print("\n========== REPAIR LOG SAMPLE ==========")
display(repair_log.head(15))


========== CLEANING RESULTS ==========
Original shape: (4000, 16)
Cleaned shape: (4000, 16)

Remaining missing values: 0

Total repair operations: 5172

========== OPERATIONS DISTRIBUTION ==========


,count
operation,
fuzzy_matching,1500
format_normalization,1500
categorical_mode_imputation,928
invalid_value_median_imputation,672
numerical_median_imputation,572



========== REPAIR LOG SAMPLE ==========


,row_index,column,original_value,repaired_value,operation
0,0,job,managemnt,management,fuzzy_matching
1,14,job,BLUE-COLLAR,blue-collar,format_normalization
2,22,job,TECHNICIAN,technician,format_normalization
3,34,job,techniian,technician,fuzzy_matching
4,36,job,ADMIN.,admin.,format_normalization
5,47,job,SERVICES,services,format_normalization
6,54,job,techncian,technician,fuzzy_matching
7,66,job,TECHNICIAN,technician,format_normalization
8,87,job,MANAGEMENT,management,format_normalization
9,94,job,studen,student,fuzzy_matching


In [7]:

# Load Ground Truth for evaluation only

mask_file = CORRUPTED_DIR / "seed_42" / "ground_truth_mask.csv"

ground_truth = pd.read_csv(mask_file)

print("========== GROUND TRUTH OVERVIEW ==========")

print("Mask shape:", ground_truth.shape)

print("\nMask columns:")
print(ground_truth.columns.tolist())

print("\nError distribution:")
display(ground_truth["error_type"].value_counts())

print("\nFirst 10 records:")
display(ground_truth.head(10))


========== GROUND TRUTH OVERVIEW ==========
Mask shape: (6000, 5)

Mask columns:
['row_index', 'column', 'error_type', 'original_value', 'corrupted_value']

Error distribution:


error_type
missing_value           1500
numerical_shift         1500
categorical_typo        1500
format_inconsistency    1500
Name: count, dtype: int64


First 10 records:


,row_index,column,error_type,original_value,corrupted_value
0,2903,poutcome,missing_value,unknown,NaN
1,3333,housing,missing_value,no,NaN
2,1666,campaign,missing_value,4,NaN
3,244,contact,missing_value,unknown,NaN
4,1234,housing,missing_value,yes,NaN
5,3995,balance,missing_value,2022,NaN
6,3488,default,missing_value,no,NaN
7,1711,age,missing_value,32,NaN
8,640,loan,missing_value,no,NaN
9,80,default,missing_value,no,NaN


In [8]:


# RULE-BASED CLEANER: GROUND TRUTH EVALUATION


# Build a reference dictionary:
# (row_index, column) -> original clean value

ground_truth_dict = {
    (int(row.row_index), row.column): row.original_value
    for row in ground_truth.itertuples(index=False)
}

# Get unique cells modified by the cleaner
repair_keys = set(
    zip(
        repair_log["row_index"].astype(int),
        repair_log["column"]
    )
)

ground_truth_keys = set(ground_truth_dict.keys())

# Cells that the cleaner attempted to repair
detected_keys = repair_keys.intersection(ground_truth_keys)

# Cells changed even though they were not corrupted
clean_cells_modified = repair_keys - ground_truth_keys


# Compare repaired value with the original value
def values_equal(a, b):

    if pd.isna(a) and pd.isna(b):
        return True

    if pd.isna(a) or pd.isna(b):
        return False

    try:
        return bool(np.isclose(float(a), float(b)))
    except (ValueError, TypeError):
        return str(a) == str(b)


# Count correctly repaired corrupted cells
correctly_repaired = 0

for row_index, column in detected_keys:

    actual_value = ground_truth_dict[(row_index, column)]
    repaired_value = df_rule_cleaned.at[row_index, column]

    if values_equal(repaired_value, actual_value):
        correctly_repaired += 1


# Incorrect repairs among corrupted cells
incorrect_repairs = len(detected_keys) - correctly_repaired

# Total unique repair operations
total_repairs = len(repair_keys)

# Metrics
repair_precision = (
    correctly_repaired / total_repairs
    if total_repairs else 0
)

repair_recall = (
    correctly_repaired / len(ground_truth_keys)
    if ground_truth_keys else 0
)

repair_f1 = (
    2 * repair_precision * repair_recall
    / (repair_precision + repair_recall)
    if repair_precision + repair_recall else 0
)

detection_precision = (
    len(detected_keys) / total_repairs
    if total_repairs else 0
)

detection_recall = (
    len(detected_keys) / len(ground_truth_keys)
    if ground_truth_keys else 0
)

# Display results
metrics = pd.DataFrame({
    "Metric": [
        "Total corrupted cells",
        "Total cells modified",
        "Corrupted cells detected",
        "Correctly repaired cells",
        "Incorrect repairs on corrupted cells",
        "Clean cells modified (false repairs)",
        "Repair Precision",
        "Repair Recall",
        "Repair F1",
        "Detection Precision",
        "Detection Recall"
    ],
    "Value": [
        len(ground_truth_keys),
        total_repairs,
        len(detected_keys),
        correctly_repaired,
        incorrect_repairs,
        len(clean_cells_modified),
        round(repair_precision, 4),
        round(repair_recall, 4),
        round(repair_f1, 4),
        round(detection_precision, 4),
        round(detection_recall, 4)
    ]
})

display(metrics)


,Metric,Value
0,Total corrupted cells,6000.0000
1,Total cells modified,5172.0000
2,Corrupted cells detected,5172.0000
3,Correctly repaired cells,4004.0000
4,Incorrect repairs on corrupted cells,1168.0000
5,Clean cells modified (false repairs),0.0000
6,Repair Precision,0.7742
7,Repair Recall,0.6673
8,Repair F1,0.7168
9,Detection Precision,1.0000


In [9]:


# PER-ERROR-TYPE EVALUATION


type_results = []

for error_type, group in ground_truth.groupby("error_type"):

    type_keys = set(
        zip(group["row_index"].astype(int), group["column"])
    )

    touched = type_keys.intersection(repair_keys)

    correct = 0

    for row_index, column in touched:
        original_value = ground_truth_dict[(row_index, column)]
        repaired_value = df_rule_cleaned.at[row_index, column]

        if values_equal(repaired_value, original_value):
            correct += 1

    total = len(type_keys)
    missed = total - len(touched)

    type_results.append({
        "Error Type": error_type,
        "Total": total,
        "Detected": len(touched),
        "Missed": missed,
        "Correctly Repaired": correct,
        "Incorrect Repairs": len(touched) - correct,
        "Repair Precision": round(correct / len(touched), 4)
            if touched else 0,
        "Repair Recall": round(correct / total, 4)
    })

type_results_df = pd.DataFrame(type_results)

display(type_results_df)


,Error Type,Total,Detected,Missed,Correctly Repaired,Incorrect Repairs,Repair Precision,Repair Recall
0,categorical_typo,1500,1500,0,1500,0,1.0000,1.0000
1,format_inconsistency,1500,1500,0,1500,0,1.0000,1.0000
2,missing_value,1500,1500,0,751,749,0.5007,0.5007
3,numerical_shift,1500,672,828,253,419,0.3765,0.1687


In [10]:

# RUN RULE-BASED CLEANER ON ALL 3 SEEDS
# Corrected evaluation + preserve previous results

all_seed_metrics = []
all_type_metrics = []

def values_equal(a, b, column):

    if pd.isna(a) and pd.isna(b):
        return True

    if pd.isna(a) or pd.isna(b):
        return False

    # Numeric comparison for numerical columns only
    if column in NUMERICAL_COLS:
        try:
            return bool(np.isclose(float(a), float(b)))
        except (ValueError, TypeError):
            return False

    # Exact comparison for categorical columns
    return str(a) == str(b)


for seed in SEEDS:

    print(f"Processing seed {seed}...")

    # Load corrupted data and ground-truth mask
    seed_dir = CORRUPTED_DIR / f"seed_{seed}"

    corrupted = pd.read_csv(
        seed_dir / "corrupted_train.csv"
    )

    mask = pd.read_csv(
        seed_dir / "ground_truth_mask.csv"
    )

    # Apply the unchanged rule-based cleaner
    cleaned, log = rule_based_clean(corrupted)

    # Ground-truth lookup
    gt_dict = {
        (int(row.row_index), row.column): row.original_value
        for row in mask.itertuples(index=False)
    }

    repair_keys_seed = set(
        zip(log["row_index"].astype(int), log["column"])
    )

    gt_keys = set(gt_dict.keys())

    detected = repair_keys_seed.intersection(gt_keys)
    clean_modified = repair_keys_seed - gt_keys

    # Evaluate exact restoration
    correct = 0

    for row_index, column in detected:

        if values_equal(
            cleaned.at[row_index, column],
            gt_dict[(row_index, column)],
            column
        ):
            correct += 1

    total_modified = len(repair_keys_seed)
    total_corrupted = len(gt_keys)

    incorrect = len(detected) - correct

    precision = (
        correct / total_modified
        if total_modified else 0
    )

    recall = (
        correct / total_corrupted
        if total_corrupted else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    detection_precision = (
        len(detected) / total_modified
        if total_modified else 0
    )

    detection_recall = (
        len(detected) / total_corrupted
        if total_corrupted else 0
    )

    all_seed_metrics.append({
        "Seed": seed,
        "Total Corrupted": total_corrupted,
        "Total Modified": total_modified,
        "Detected": len(detected),
        "Correctly Repaired": correct,
        "Incorrect Repairs": incorrect,
        "Clean Cells Modified": len(clean_modified),
        "Repair Precision": precision,
        "Repair Recall": recall,
        "Repair F1": f1,
        "Detection Precision": detection_precision,
        "Detection Recall": detection_recall
    })

    # Evaluate each corruption type
    for error_type, group in mask.groupby("error_type"):

        type_keys = set(
            zip(
                group["row_index"].astype(int),
                group["column"]
            )
        )

        touched = type_keys.intersection(repair_keys_seed)

        type_correct = 0

        for row_index, column in touched:

            if values_equal(
                cleaned.at[row_index, column],
                gt_dict[(row_index, column)],
                column
            ):
                type_correct += 1

        type_total = len(type_keys)

        type_precision = (
            type_correct / len(touched)
            if touched else 0
        )

        type_recall = (
            type_correct / type_total
            if type_total else 0
        )

        all_type_metrics.append({
            "Seed": seed,
            "Error Type": error_type,
            "Total": type_total,
            "Detected": len(touched),
            "Missed": type_total - len(touched),
            "Correctly Repaired": type_correct,
            "Incorrect Repairs": len(touched) - type_correct,
            "Repair Precision": type_precision,
            "Repair Recall": type_recall
        })

    # Save cleaned data and repair log
    output_dir = RESULTS_DIR / f"seed_{seed}"
    output_dir.mkdir(parents=True, exist_ok=True)

    cleaned.to_csv(
        output_dir / "rule_based_cleaned_corrected.csv",
        index=False
    )

    log.to_csv(
        output_dir / "rule_based_repair_log_corrected.csv",
        index=False
    )

# Build result tables
overall_results = pd.DataFrame(all_seed_metrics)
type_results_all = pd.DataFrame(all_type_metrics)

# Save corrected result tables without overwriting old files
overall_results.to_csv(
    RESULTS_DIR / "rule_based_overall_by_seed_corrected.csv",
    index=False
)

type_results_all.to_csv(
    RESULTS_DIR / "rule_based_by_error_type_corrected.csv",
    index=False
)

print("\nOVERALL RESULTS")
display(overall_results.round(4))

print("\nRESULTS BY ERROR TYPE")
display(type_results_all.round(4))

print("\nCorrected results saved successfully.")


Processing seed 42...
Processing seed 123...
Processing seed 2026...

OVERALL RESULTS


,Seed,Total Corrupted,Total Modified,Detected,Correctly Repaired,Incorrect Repairs,Clean Cells Modified,Repair Precision,Repair Recall,Repair F1,Detection Precision,Detection Recall
0,42,6000,5172,5172,4004,1168,0,0.7742,0.6673,0.7168,1.0,0.8620
1,123,6000,5170,5170,4030,1140,0,0.7795,0.6717,0.7216,1.0,0.8617
2,2026,6000,5187,5187,4008,1179,0,0.7727,0.6680,0.7165,1.0,0.8645



RESULTS BY ERROR TYPE


,Seed,Error Type,Total,Detected,Missed,Correctly Repaired,Incorrect Repairs,Repair Precision,Repair Recall
0,42,categorical_typo,1500,1500,0,1500,0,1.0000,1.0000
1,42,format_inconsistency,1500,1500,0,1500,0,1.0000,1.0000
2,42,missing_value,1500,1500,0,751,749,0.5007,0.5007
3,42,numerical_shift,1500,672,828,253,419,0.3765,0.1687
4,123,categorical_typo,1500,1500,0,1500,0,1.0000,1.0000
5,123,format_inconsistency,1500,1500,0,1500,0,1.0000,1.0000
6,123,missing_value,1500,1500,0,764,736,0.5093,0.5093
7,123,numerical_shift,1500,670,830,266,404,0.3970,0.1773
8,2026,categorical_typo,1500,1500,0,1500,0,1.0000,1.0000
9,2026,format_inconsistency,1500,1500,0,1500,0,1.0000,1.0000



Corrected results saved successfully.


In [11]:

# Check available result variables

for name in [
    "overall_results",
    "overall_results_df",
    "error_type_results",
    "error_type_results_df",
    "type_results_df"
]:
    if name in globals():
        obj = globals()[name]
        print(name, "->", type(obj).__name__, getattr(obj, "shape", ""))


overall_results -> DataFrame (3, 12)
type_results_df -> DataFrame (4, 8)


In [12]:


# SAVE RULE-BASED RESULTS

from pathlib import Path

RESULTS_DIR = PROJECT_ROOT / "results" / "cleaning"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save overall results
overall_results.to_csv(
    RESULTS_DIR / "rule_based_overall_results.csv",
    index=False
)

# Save per-error-type results
type_results_all.to_csv(
    RESULTS_DIR / "rule_based_error_type_results.csv",
    index=False
)

# Save summary statistics
summary_metrics.to_csv(
    RESULTS_DIR / "rule_based_summary_statistics.csv"
)

print("Results saved successfully.")

for file in RESULTS_DIR.glob("rule_based_*.csv"):
    print(file.name, "-", file.stat().st_size, "bytes")


NameError: name 'summary_metrics' is not defined

In [13]:
import requests

response = requests.get(
    "http://localhost:11434/api/tags",
    timeout=10
)

response.raise_for_status()

models = response.json()["models"]

for model in models:
    print(model["name"])

qwen3:4b


In [14]:
import requests

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:4b",
        "messages": [
            {
                "role": "user",
                "content": "Reply with exactly: LLM_CONNECTED"
            }
        ],
        "stream": False,
        "think": False,
        "options": {
            "temperature": 0
        }
    },
    timeout=120
)

response.raise_for_status()

result = response.json()

print("Model:", result["model"])
print("Response:", result["message"]["content"])
print("Prompt tokens:", result.get("prompt_eval_count"))
print("Output tokens:", result.get("eval_count"))

Model: qwen3:4b
Response: Hmm, the user wants me to reply with exactly "LLM_CONNECTED". This seems like a very specific request where they're testing if I can follow precise instructions.

I notice they're being very clear about the output format - they want no extra text, just those exact characters. This might be for some kind of system integration test or API check. 

The user could be a developer verifying my connection status, or maybe someone testing how I handle constrained responses. Since they specified "exactly", I shouldn't add any spaces or punctuation. 

I recall that in some technical contexts, "LLM_CONNECTED" is used as a status indicator. Better make sure I don't accidentally add a newline or anything. 

Okay, I'll just output "LLM_CONNECTED" as requested. No extra words, no explanations - strictly what they asked for. 

*checks mental note* Yep, this is straightforward. User seems technical and precise, so I'll match that precision in response.
</think>

LLM_CONNECTED


In [15]:
print("Ground Truth original_value dtype:")
print(ground_truth["original_value"].dtype)

print("\nSample original values and their Python types:")

for value in ground_truth["original_value"].dropna().head(15):
    print(repr(value), type(value).__name__)

print("\nNumeric columns comparison:")

for col in NUMERICAL_COLS:
    sample = ground_truth[
        ground_truth["column"] == col
    ]["original_value"].dropna().head(3)

    print(f"\n{col}:")
    for value in sample:
        print(repr(value), type(value).__name__)

Ground Truth original_value dtype:
str

Sample original values and their Python types:
'unknown' str
'no' str
'4' str
'unknown' str
'yes' str
'2022' str
'no' str
'32' str
'no' str
'no' str
'no' str
'services' str
'34' str
'married' str
'-1' str

Numeric columns comparison:

age:
'32' str
'34' str
'40' str

balance:
'2022' str
'4136' str
'538' str

day:
'25' str
'23' str
'29' str

campaign:
'4' str
'4' str
'2' str

pdays:
'-1' str
'-1' str
'-1' str

previous:
'1' str
'0' str
'0' str


In [16]:

print("Old vs corrected comparison:")

for original, repaired, column in [
    ("32", 32.0, "age"),
    ("2022", 2022.0, "balance"),
    ("married", "married", "marital"),
    ("yes", "no", "housing")
]:
    print(
        column,
        "| Old:",
        values_equal(repaired, original, column),
        "| Corrected:",
        values_equal_corrected(
            repaired, original, column
        )
    )


Old vs corrected comparison:


NameError: name 'values_equal_corrected' is not defined

In [42]:
# SUMMARY STATISTICS ACROSS ALL 3 SEEDS

summary_metrics = overall_results[
    [
        "Repair Precision",
        "Repair Recall",
        "Repair F1",
        "Detection Precision",
        "Detection Recall"
    ]
].agg(["mean", "std"]).T

summary_metrics.columns = ["Mean", "SD"]

print("RULE-BASED BASELINE: MEAN AND STANDARD DEVIATION")

display(summary_metrics.round(4))

summary_metrics.to_csv(
    RESULTS_DIR / "rule_based_summary_statistics.csv"
)

print("Summary statistics saved successfully.")

RULE-BASED BASELINE: MEAN AND STANDARD DEVIATION


,Mean,SD
Repair Precision,0.7755,0.0036
Repair Recall,0.6690,0.0023
Repair F1,0.7183,0.0028
Detection Precision,1.0000,0.0000
Detection Recall,0.8627,0.0015


Summary statistics saved successfully.


In [43]:

# Inspect categorical values and their frequencies

categorical_columns = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome",
]

for column in categorical_columns:

    print("\n" + "=" * 60)
    print(f"COLUMN: {column}")
    print("=" * 60)

    value_counts = (
        df_corrupted[column]
        .value_counts(dropna=False)
    )

    print(value_counts.to_string())



COLUMN: job
job
blue-collar        785
management         741
technician         560
admin.             424
services           323
retired            182
unemployed         126
housemaid          116
self-employed      115
NaN                109
entrepreneur       101
student             67
 MANAGEMENT         36
 BLUE-COLLAR        31
 TECHNICIAN         31
 SERVICES           20
unknown             18
 ADMIN.             15
 RETIRED            10
manageent            8
bue-collar           8
 SELF-EMPLOYED       8
techniian            6
anagement            6
blue-colar           6
amin.                5
manaement            5
mnagement            5
 UNEMPLOYED          5
blue-ollar           5
 ENTREPRENEUR        5
managment            5
technicia            5
techncian            4
lue-collar           4
srvices              4
technican            4
dmin.                4
sevices              4
managemnt            3
housemad             3
admin                3
blue-cllar       

In [44]:
ALLOWED_VALUES = {
    "job": [
        "admin.", "blue-collar", "entrepreneur", "housemaid",
        "management", "retired", "self-employed", "services",
        "student", "technician", "unemployed", "unknown"
    ],
    "marital": ["divorced", "married", "single"],
    "education": ["primary", "secondary", "tertiary", "unknown"],
    "default": ["yes", "no"],
    "housing": ["yes", "no"],
    "loan": ["yes", "no"],
    "contact": ["cellular", "telephone", "unknown"],
    "month": [
        "jan", "feb", "mar", "apr", "may", "jun",
        "jul", "aug", "sep", "oct", "nov", "dec"
    ],
    "poutcome": ["failure", "other", "success", "unknown"]
}

UNKNOWN_VALUES = {
    "job": ["unknown"],
    "education": ["unknown"],
    "contact": ["unknown"],
    "poutcome": ["unknown"]
}

for column, allowed in ALLOWED_VALUES.items():
    values = df_corrupted[column].dropna().astype(str).str.strip().str.lower()

    suspicious = sorted(set(values) - set(allowed))

    print(f"\n{column}")
    print("Allowed values:", len(allowed))
    print("Suspicious unique values:", len(suspicious))
    print("Examples:", suspicious[:15])


job
Allowed values: 12
Suspicious unique values: 61
Examples: ['admi.', 'admin', 'admn.', 'amin.', 'anagement', 'ble-collar', 'blu-collar', 'blue-cllar', 'blue-colar', 'blue-colla', 'blue-collr', 'blue-ollar', 'bluecollar', 'bue-collar', 'dmin.']

marital
Allowed values: 3
Suspicious unique values: 20
Examples: ['arried', 'diorced', 'divoced', 'divorcd', 'divorce', 'divored', 'divrced', 'dvorced', 'ingle', 'ivorced', 'maried', 'marred', 'marrid', 'marrie', 'mrried']

education
Allowed values: 4
Suspicious unique values: 29
Examples: ['econdary', 'ertiary', 'nknown', 'pimary', 'priary', 'primar', 'primay', 'primry', 'prmary', 'rimary', 'scondary', 'secndary', 'secodary', 'seconary', 'secondar']

default
Allowed values: 2
Suspicious unique values: 2
Examples: ['nox', 'yesx']

housing
Allowed values: 2
Suspicious unique values: 2
Examples: ['nox', 'yesx']

loan
Allowed values: 2
Suspicious unique values: 2
Examples: ['nox', 'yesx']

contact
Allowed values: 3
Suspicious unique values: 19


In [45]:
import pandas as pd

suspicious_report = []

for column, allowed in ALLOWED_VALUES.items():

    values = (
        df_corrupted[column]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )

    counts = values.value_counts()

    for value, count in counts.items():

        if value not in allowed:

            suspicious_report.append({
                "column": column,
                "suspicious_value": value,
                "frequency": int(count)
            })

suspicious_df = pd.DataFrame(suspicious_report)

suspicious_df = suspicious_df.sort_values(
    ["column", "frequency"],
    ascending=[True, False]
)

display(suspicious_df.head(30))

print("Total suspicious unique values:", len(suspicious_df))
print(
    "Total suspicious cells:",
    suspicious_df["frequency"].sum()
)

suspicious_df.to_csv(
    RESULTS_DIR / "suspicious_values_seed_42.csv",
    index=False
)

,column,suspicious_value,frequency
116,contact,celular,25
117,contact,ellular,19
118,contact,cllular,18
119,contact,celluar,12
120,contact,unkown,11
121,contact,cellula,11
122,contact,celllar,11
123,contact,nknown,11
124,contact,uknown,9
125,contact,cellulr,8


Total suspicious unique values: 167
Total suspicious cells: 1500


In [46]:
import sys
import importlib

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import agents.llm_planner as llm_planner

importlib.reload(llm_planner)

propose_corrections = llm_planner.propose_corrections

print("LLM Planner imported successfully!")

LLM Planner imported successfully!


In [47]:
print(propose_corrections)

<function propose_corrections at 0x00000269EF0B0D60>


In [48]:
import importlib
import agents.llm_planner as llm_planner

importlib.reload(llm_planner)

propose_corrections = llm_planner.propose_corrections

print("LLM Planner updated successfully!")

LLM Planner updated successfully!


In [49]:
test_column = "contact"

test_values = (
    suspicious_df[
        suspicious_df["column"] == test_column
    ]
    .head(20)
    .to_dict(orient="records")
)

test_result = propose_corrections(
    suspicious_values=test_values,
    allowed_values=ALLOWED_VALUES[test_column]
)

print(json.dumps(
    test_result,
    indent=2,
    ensure_ascii=False
))

{
  "response": {
    "proposals": [
      {
        "original_value": "celular",
        "suggested_value": "cellular",
        "reason": "Typo for 'cellular' (missing 'l')",
        "confidence": 0.9
      },
      {
        "original_value": "ellular",
        "suggested_value": "cellular",
        "reason": "Typo for 'cellular' (missing 'c')",
        "confidence": 0.8
      },
      {
        "original_value": "cllular",
        "suggested_value": "cellular",
        "reason": "Typo for 'cellular' (extra 'l')",
        "confidence": 0.7
      },
      {
        "original_value": "celluar",
        "suggested_value": "cellular",
        "reason": "Typo for 'cellular' (missing 'l')",
        "confidence": 0.8
      },
      {
        "original_value": "unkown",
        "suggested_value": "unknown",
        "reason": "Typo for 'unknown' (missing 'n')",
        "confidence": 0.9
      },
      {
        "original_value": "cellula",
        "suggested_value": "cellular",
        "reaso

In [50]:
test_column = "contact"

test_values_5 = (
    suspicious_df[
        suspicious_df["column"] == test_column
    ]
    .head(5)
    .to_dict(orient="records")
)

test_result_5 = propose_corrections(
    suspicious_values=test_values_5,
    allowed_values=ALLOWED_VALUES[test_column]
)

print(json.dumps(
    test_result_5,
    indent=2,
    ensure_ascii=False
))

{
  "response": {
    "proposals": [
      {
        "original_value": "celular",
        "suggested_value": "cellular",
        "reason": "Typo: missing 'l' in 'cellular'",
        "confidence": 0.9
      },
      {
        "original_value": "ellular",
        "suggested_value": "cellular",
        "reason": "Typo: missing 'c' at the start",
        "confidence": 0.8
      },
      {
        "original_value": "cllular",
        "suggested_value": "cellular",
        "reason": "Typo: extra 'l' in 'cellular'",
        "confidence": 0.7
      },
      {
        "original_value": "celluar",
        "suggested_value": "cellular",
        "reason": "Typo: missing 'l' in 'cellular'",
        "confidence": 0.8
      },
      {
        "original_value": "unkown",
        "suggested_value": "unknown",
        "reason": "Typo: missing 'n' in 'unknown'",
        "confidence": 0.9
      }
    ]
  },
  "runtime_seconds": 15.46177650000027,
  "prompt_tokens": 268,
  "completion_tokens": 254,
  "tota

In [51]:
import sys
import importlib

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import agents.proposal_validator as proposal_validator
importlib.reload(proposal_validator)

validate_proposals = proposal_validator.validate_proposals

print("Proposal Validator imported successfully!")

Proposal Validator imported successfully!


In [52]:
valid_proposals, rejected_proposals = validate_proposals(
    proposals=test_result_5["response"]["proposals"],
    suspicious_values=test_values_5,
    allowed_values=ALLOWED_VALUES["contact"]
)

print("Valid proposals:", len(valid_proposals))
print("Rejected proposals:", len(rejected_proposals))

print("\nValid:")
print(json.dumps(valid_proposals, indent=2, ensure_ascii=False))

print("\nRejected:")
print(json.dumps(rejected_proposals, indent=2, ensure_ascii=False))

Valid proposals: 5
Rejected proposals: 0

Valid:
[
  {
    "original_value": "celular",
    "suggested_value": "cellular",
    "reason": "Typo: missing 'l' in 'cellular'",
    "confidence": 0.9
  },
  {
    "original_value": "ellular",
    "suggested_value": "cellular",
    "reason": "Typo: missing 'c' at the start",
    "confidence": 0.8
  },
  {
    "original_value": "cllular",
    "suggested_value": "cellular",
    "reason": "Typo: extra 'l' in 'cellular'",
    "confidence": 0.7
  },
  {
    "original_value": "celluar",
    "suggested_value": "cellular",
    "reason": "Typo: missing 'l' in 'cellular'",
    "confidence": 0.8
  },
  {
    "original_value": "unkown",
    "suggested_value": "unknown",
    "reason": "Typo: missing 'n' in 'unknown'",
    "confidence": 0.9
  }
]

Rejected:
[]


In [53]:
import agents.safe_executor as safe_executor
import importlib

importlib.reload(safe_executor)

execute_categorical_proposals = (
    safe_executor.execute_categorical_proposals
)

df_autonomous, autonomous_log = execute_categorical_proposals(
    dataframe=df_corrupted.copy(),
    column="contact",
    valid_proposals=valid_proposals
)

print("Original rows:", len(df_corrupted))
print("Cleaned rows:", len(df_autonomous))
print("Executed operations:", len(autonomous_log))
print("Affected cells:", sum(
    item["affected_rows"] for item in autonomous_log
))

print("\nExecution log:")
print(pd.DataFrame(autonomous_log).to_string(index=False))

Original rows: 4000
Cleaned rows: 4000
Executed operations: 5
Affected cells: 85

Execution log:
 column original_value suggested_value  affected_rows                          reason  confidence
contact        celular        cellular             25 Typo: missing 'l' in 'cellular'         0.9
contact        ellular        cellular             19  Typo: missing 'c' at the start         0.8
contact        cllular        cellular             18   Typo: extra 'l' in 'cellular'         0.7
contact        celluar        cellular             12 Typo: missing 'l' in 'cellular'         0.8
contact         unkown         unknown             11  Typo: missing 'n' in 'unknown'         0.9


In [54]:
import agents.hitl_controller as hitl_controller
import importlib

importlib.reload(hitl_controller)

collect_human_decisions = (
    hitl_controller.collect_human_decisions
)

test_decisions = {
    0: "approve",
    1: "reject",
    2: "approve",
    3: "approve",
    4: "reject"
}

approved, rejected, decision_log = collect_human_decisions(
    valid_proposals,
    test_decisions
)

print("Approved:", len(approved))
print("Rejected:", len(rejected))

print("\nDecision log:")
print(pd.DataFrame(decision_log).to_string(index=False))

Approved: 3
Rejected: 2

Decision log:
 operation_id  column original_value suggested_value decision
            1 contact        celular        cellular  approve
            2 contact        ellular        cellular   reject
            3 contact        cllular        cellular  approve
            4 contact        celluar        cellular  approve
            5 contact         unkown         unknown   reject


In [55]:
df_hitl, hitl_execution_log = execute_categorical_proposals(
    dataframe=df_corrupted.copy(),
    column="contact",
    valid_proposals=valid_proposals,
    approved_proposals=approved
)

print("Original rows:", len(df_corrupted))
print("HITL rows:", len(df_hitl))

print("Approved operations executed:", len(hitl_execution_log))

print(
    "Affected cells:",
    sum(item["affected_rows"] for item in hitl_execution_log)
)

print("\nExecution log:")
print(pd.DataFrame(hitl_execution_log).to_string(index=False))

Original rows: 4000
HITL rows: 4000
Approved operations executed: 3
Affected cells: 55

Execution log:
 column original_value suggested_value  affected_rows                          reason  confidence
contact        celular        cellular             25 Typo: missing 'l' in 'cellular'         0.9
contact        cllular        cellular             18   Typo: extra 'l' in 'cellular'         0.7
contact        celluar        cellular             12 Typo: missing 'l' in 'cellular'         0.8


In [56]:
from pathlib import Path
import pandas as pd
import numpy as np

# Load evaluation references
clean_reference = pd.read_csv(
    PROJECT_ROOT / "data" / "clean" / "train_clean_reference.csv",
    keep_default_na=False
)

ground_truth = pd.read_csv(
    PROJECT_ROOT / "data" / "corrupted" / "seed_42" / "ground_truth_mask.csv",
    keep_default_na=False
)

# Evaluation function
def evaluate_column(
    original_corrupted,
    cleaned,
    clean_reference,
    ground_truth,
    column
):
    before = original_corrupted[column].astype("string")
    after = cleaned[column].astype("string")
    reference = clean_reference[column].astype("string")

    changed = (
        before.ne(after).fillna(False)
    )

    # Ground-truth corrupted rows for this column
    corrupted_rows = set(
        ground_truth.loc[
            ground_truth["column"] == column,
            "row_index"
        ].astype(int)
    )

    is_corrupted = pd.Series(
        original_corrupted.index.isin(corrupted_rows),
        index=original_corrupted.index
    )

    correctly_restored = changed & is_corrupted & after.eq(reference).fillna(False)

    incorrectly_repaired = changed & is_corrupted & ~after.eq(reference).fillna(False)

    harmful_clean_changes = changed & ~is_corrupted

    total_corrupted = len(corrupted_rows)
    total_changed = int(changed.sum())
    correct_count = int(correctly_restored.sum())
    incorrect_count = int(incorrectly_repaired.sum())
    harmful_count = int(harmful_clean_changes.sum())

    detected_corrupted = correct_count + incorrect_count

    precision = correct_count / total_changed if total_changed else 0
    recall = correct_count / total_corrupted if total_corrupted else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    return {
        "column": column,
        "total_corrupted": total_corrupted,
        "total_changed": total_changed,
        "correct_repairs": correct_count,
        "incorrect_repairs": incorrect_count,
        "harmful_clean_changes": harmful_count,
        "missed_corruptions": total_corrupted - detected_corrupted,
        "repair_precision": precision,
        "repair_recall": recall,
        "repair_f1": f1
    }


# Evaluate both approaches
autonomous_metrics = evaluate_column(
    df_corrupted,
    df_autonomous,
    clean_reference,
    ground_truth,
    "contact"
)

hitl_metrics = evaluate_column(
    df_corrupted,
    df_hitl,
    clean_reference,
    ground_truth,
    "contact"
)

comparison = pd.DataFrame([
    {"approach": "Autonomous", **autonomous_metrics},
    {"approach": "HITL", **hitl_metrics}
])

display(comparison.T)

,0,1
approach,Autonomous,HITL
column,contact,contact
total_corrupted,407,407
total_changed,85,55
correct_repairs,85,55
incorrect_repairs,0,0
harmful_clean_changes,0,0
missed_corruptions,322,352
repair_precision,1.0,1.0
repair_recall,0.208845,0.135135


In [58]:
from IPython.display import display
import pandas as pd
import time

review_decisions = []
review_start = time.perf_counter()

for i, proposal in enumerate(valid_proposals):
    print(f"\n {i+1}/{len(valid_proposals)}")
    print(f"Column: {proposal.get('column', 'contact')}")
    print(f"Original: {proposal['original_value']}")
    print(f"Suggested: {proposal['suggested_value']}")
    print(f"Reason: {proposal.get('reason')}")

    while True:
        decision = input("Approve or Reject? (a/r): ").strip().lower()
        if decision in ("a", "r"):
            break
        print("Please enter a or r.")

    review_decisions.append({
        **proposal,
        "decision": "
        " if decision == "a" else "reject"
    })

review_time = time.perf_counter() - review_start

print("\nReview completed.")
print("Approved:", sum(x["decision"] == "approve" for x in review_decisions))
print("Rejected:", sum(x["decision"] == "reject" for x in review_decisions))
print(f"Review time: {review_time:.2f} seconds")

SyntaxError: unterminated string literal (detected at line 23) (831946470.py, line 23)

In [59]:
approved_proposals = [
    {**item, "column": "contact"}
    for item in review_decisions
    if item["decision"] == "approve"
]

rejected_proposals = [
    item for item in review_decisions
    if item["decision"] == "reject"
]

hitl_cleaned, hitl_execution_log = execute_categorical_proposals(
    dataframe=df_corrupted,
    column="contact",
    valid_proposals=approved_proposals
)

print("Approved operations:", len(approved_proposals))
print("Rejected operations:", len(rejected_proposals))
print("Executed operations:", len(hitl_execution_log))
print("Affected cells:", sum(x["affected_rows"] for x in hitl_execution_log))
print("Review time:", round(review_time, 2), "seconds")

NameError: name 'review_decisions' is not defined

In [60]:
hitl_metrics = evaluate_column(
    original_corrupted=df_corrupted,
    cleaned=hitl_cleaned,
    clean_reference=clean_reference,
    ground_truth=ground_truth,
    column="contact"
)

pd.DataFrame([hitl_metrics])

NameError: name 'hitl_cleaned' is not defined

In [61]:
from pathlib import Path
import json

results_dir = PROJECT_ROOT / "results" / "operational"
results_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame([hitl_metrics]).to_csv(
    results_dir / "hitl_pilot_metrics_seed_42_contact.csv",
    index=False
)

pd.DataFrame(review_decisions).to_csv(
    results_dir / "hitl_pilot_decisions_seed_42_contact.csv",
    index=False
)

pd.DataFrame(hitl_execution_log).to_csv(
    results_dir / "hitl_pilot_execution_log_seed_42_contact.csv",
    index=False
)

with open(results_dir / "hitl_pilot_review_time_seed_42.json", "w") as f:
    json.dump({"review_time_seconds": review_time}, f, indent=4)

print("HITL pilot results saved successfully.")

NameError: name 'review_decisions' is not defined

In [62]:
all_suspicious_values = {}

for column, group in suspicious_df.groupby("column"):
    all_suspicious_values[column] = [
        {
            "suspicious_value": row["suspicious_value"],
            "frequency": int(row["frequency"])
        }
        for _, row in group.iterrows()
    ]

print("Columns:", len(all_suspicious_values))
print("Total suspicious values:", sum(
    len(values) for values in all_suspicious_values.values()
))

for column, values in all_suspicious_values.items():
    print(f"{column}: {len(values)} values")

Columns: 9
Total suspicious values: 167
contact: 19 values
default: 2 values
education: 29 values
housing: 2 values
job: 61 values
loan: 2 values
marital: 20 values
month: 12 values
poutcome: 20 values


In [63]:
all_llm_proposals = {}
all_llm_metrics = {}

for column, values in all_suspicious_values.items():
    print(f"\nProcessing: {column} ({len(values)} values)")

    allowed = ALLOWED_VALUES[column]

    column_proposals = []
    column_metrics = []

    for start in range(0, len(values), 5):
        batch = values[start:start + 5]

        result = propose_corrections(batch, allowed)

        proposals = result["response"].get("proposals", [])

        valid, rejected = validate_proposals(
            proposals,
            batch,
            allowed
        )

        column_proposals.extend(valid)

        column_metrics.append({
            "batch": start // 5 + 1,
            "input_values": len(batch),
            "returned": len(proposals),
            "valid": len(valid),
            "rejected": len(rejected),
            "runtime_seconds": result["runtime_seconds"],
            "prompt_tokens": result["prompt_tokens"],
            "completion_tokens": result["completion_tokens"]
        })

    all_llm_proposals[column] = column_proposals
    all_llm_metrics[column] = column_metrics

    print("Valid proposals:", len(column_proposals))
    print("Batches:", len(column_metrics))

print("\nAll columns processed.")


Processing: contact (19 values)
Valid proposals: 19
Batches: 4

Processing: default (2 values)
Valid proposals: 2
Batches: 1

Processing: education (29 values)
Valid proposals: 29
Batches: 6

Processing: housing (2 values)
Valid proposals: 2
Batches: 1

Processing: job (61 values)
Valid proposals: 61
Batches: 13

Processing: loan (2 values)
Valid proposals: 2
Batches: 1

Processing: marital (20 values)
Valid proposals: 19
Batches: 4

Processing: month (12 values)
Valid proposals: 12
Batches: 3

Processing: poutcome (20 values)
Valid proposals: 20
Batches: 4

All columns processed.


In [64]:
import pandas as pd
import json

results_dir = PROJECT_ROOT / "results" / "operational"
results_dir.mkdir(parents=True, exist_ok=True)

# Collecting suggestions
proposal_rows = []

for column, proposals in all_llm_proposals.items():
    for proposal in proposals:
        proposal_rows.append({
            "column": column,
            **proposal
        })

proposals_df = pd.DataFrame(proposal_rows)

# Collecting operational metrics
metrics_rows = []

for column, batches in all_llm_metrics.items():
    for batch in batches:
        metrics_rows.append({
            "column": column,
            **batch
        })

metrics_df = pd.DataFrame(metrics_rows)

# Save results
proposals_df.to_csv(
    results_dir / "llm_all_proposals_seed_42.csv",
    index=False
)

metrics_df.to_csv(
    results_dir / "llm_operational_metrics_seed_42.csv",
    index=False
)

# Operation Summary
print("Total proposals:", len(proposals_df))
print("Total batches:", len(metrics_df))
print("Total runtime:", round(metrics_df["runtime_seconds"].sum(), 2), "seconds")
print("Total prompt tokens:", metrics_df["prompt_tokens"].sum())
print("Total completion tokens:", metrics_df["completion_tokens"].sum())

print("Missing proposals:")

proposed_values = {
    (row["column"], str(row["original_value"]).strip().lower())
    for _, row in proposals_df.iterrows()
}

missing_values = []

for column, values in all_suspicious_values.items():
    for item in values:
        key = (column, item["suspicious_value"].strip().lower())

        if key not in proposed_values:
            missing_values.append({
                "column": column,
                "suspicious_value": item["suspicious_value"]
            })

print("Missing count:", len(missing_values))
print(pd.DataFrame(missing_values))

Total proposals: 166
Total batches: 37
Total runtime: 545.14 seconds
Total prompt tokens: 8957
Total completion tokens: 9070
Missing proposals:
Missing count: 1
    column suspicious_value
0  marital          divored


In [65]:
review_df = proposals_df.copy()

review_df["decision"] = ""
review_df["reviewed"] = False

review_df = review_df.sort_values(
    ["column", "original_value"]
).reset_index(drop=True)

review_path = results_dir / "hitl_review_seed_42.csv"

review_df.to_csv(review_path, index=False)

print("Review file created:", review_path)
print("Proposals ready for review:", len(review_df))
print("Columns:", review_df["column"].nunique())

Review file created: C:\Users\ayman\OneDrive\Desktop\hitl-tabular-data-preparation\results\operational\hitl_review_seed_42.csv
Proposals ready for review: 166
Columns: 9


In [66]:

import ipywidgets as widgets
from IPython.display import display, clear_output
import time

review_index = 0
review_start = time.perf_counter()
review_records = []

title = widgets.HTML()
details = widgets.HTML()
approve_btn = widgets.Button(description="Approve", button_style="success")
reject_btn = widgets.Button(description="Reject", button_style="danger")
output = widgets.Output()

def show_proposal():
    with output:
        clear_output(wait=True)

        if review_index >= len(review_df):
            print("Review completed!")
            return

        row = review_df.iloc[review_index]

        title.value = f"<h3>Proposal {review_index + 1} / {len(review_df)}</h3>"
        details.value = f"""
        <b>Column:</b> {row['column']}<br>
        <b>Original:</b> {row['original_value']}<br>
        <b>Suggested:</b> {row['suggested_value']}<br>
        <b>Reason:</b> {row.get('reason', '')}<br>
        <b>Confidence:</b> {row.get('confidence', '')}
        """

        display(title, details, widgets.HBox([approve_btn, reject_btn]))

def record_decision(decision):
    global review_index

    row = review_df.iloc[review_index]

    review_records.append({
        "column": row["column"],
        "original_value": row["original_value"],
        "suggested_value": row["suggested_value"],
        "reason": row.get("reason", ""),
        "confidence": row.get("confidence", ""),
        "decision": decision,
        "decision_time": time.strftime("%Y-%m-%d %H:%M:%S")
    })

    review_index += 1
    show_proposal()

approve_btn.on_click(lambda b: record_decision("approve"))
reject_btn.on_click(lambda b: record_decision("reject"))

show_proposal()
display(output)

HTML(value='<h3>Proposal 1 / 166</h3>')

HTML(value="\n        <b>Column:</b> contact<br>\n        <b>Original:</b> celllar<br>\n        <b>Suggested:<…

Output()

In [67]:
import pandas as pd
import json

decisions_df = pd.DataFrame(review_records)

decisions_df.to_csv(
    results_dir / "hitl_decisions_seed_42.csv",
    index=False
)

with open(
    results_dir / "hitl_review_summary_seed_42.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump({
        "reviewer": "single human reviewer",
        "total_reviewed": len(review_records),
        "approved": 136,
        "rejected": 30,
        "review_time_seconds": review_time
    }, f, indent=4)

print("Decisions saved:", len(decisions_df))

NameError: name 'review_time' is not defined

In [68]:
decisions_df.loc[
    decisions_df["original_value"].astype(str).str.lower() == "tertiay",
    "decision"
] = "reject"

decisions_df.to_csv(
    results_dir / "hitl_decisions_seed_42.csv",
    index=False
)

print(decisions_df["decision"].value_counts())

KeyError: 'original_value'

In [69]:
approved_df = decisions_df[
    decisions_df["decision"] == "approve"
].copy()

print("Approved proposals:", len(approved_df))
print("Rejected proposals:", len(decisions_df) - len(approved_df))
print("Columns:", approved_df["column"].nunique())
print(sorted(approved_df["column"].unique()))

KeyError: 'decision'

In [70]:
import pandas as pd
from pathlib import Path

project_dir = Path(
    r"C:\Users\ayman\OneDrive\Desktop\hitl-tabular-data-preparation"
)

data_dir = project_dir / "data"
results_dir = project_dir / "results"

# Load corrupted training data
hitl_df = pd.read_csv(
    data_dir / "corrupted" / "seed_42" / "corrupted_train.csv",
    dtype=str,
    keep_default_na=False
)

execution_logs = []

for column in sorted(approved_df["column"].unique()):

    column_proposals = approved_df[
        approved_df["column"] == column
    ]

    for _, proposal in column_proposals.iterrows():

        original = str(proposal["original_value"])
        suggested = str(proposal["suggested_value"])

        mask = hitl_df[column] == original
        affected_rows = hitl_df.index[mask].tolist()

        if affected_rows:
            hitl_df.loc[mask, column] = suggested

        execution_logs.append({
            "column": column,
            "original_value": original,
            "suggested_value": suggested,
            "affected_cells": len(affected_rows),
            "affected_rows": affected_rows,
            "status": "executed" if affected_rows else "no_matching_value"
        })

# Save results
hitl_output = data_dir / "clean" / "hitl_cleaned_seed_42.csv"
hitl_output.parent.mkdir(parents=True, exist_ok=True)
hitl_df.to_csv(hitl_output, index=False)

execution_log_df = pd.DataFrame(execution_logs)

execution_log_df.to_json(
    results_dir / "operational" / "hitl_execution_log_seed_42.json",
    orient="records",
    indent=4
)

print("Approved proposals:", len(approved_df))
print("Executed proposals:", (execution_log_df["status"] == "executed").sum())
print("Unmatched proposals:", (execution_log_df["status"] == "no_matching_value").sum())
print("Total affected cells:", execution_log_df["affected_cells"].sum())
print("Saved:", hitl_output)

NameError: name 'approved_df' is not defined

In [71]:
print("Clean reference files:")
print(list((data_dir / "clean").glob("*")))

print("\nGround truth files:")
print(list((data_dir / "corrupted" / "seed_42").glob("*")))

Clean reference files:
[WindowsPath('C:/Users/ayman/OneDrive/Desktop/hitl-tabular-data-preparation/data/clean/.ipynb_checkpoints'), WindowsPath('C:/Users/ayman/OneDrive/Desktop/hitl-tabular-data-preparation/data/clean/autonomous_cleaned_seed_123.csv'), WindowsPath('C:/Users/ayman/OneDrive/Desktop/hitl-tabular-data-preparation/data/clean/autonomous_cleaned_seed_2026.csv'), WindowsPath('C:/Users/ayman/OneDrive/Desktop/hitl-tabular-data-preparation/data/clean/autonomous_cleaned_seed_42.csv'), WindowsPath('C:/Users/ayman/OneDrive/Desktop/hitl-tabular-data-preparation/data/clean/hitl_cleaned_seed_123.csv'), WindowsPath('C:/Users/ayman/OneDrive/Desktop/hitl-tabular-data-preparation/data/clean/hitl_cleaned_seed_2026.csv'), WindowsPath('C:/Users/ayman/OneDrive/Desktop/hitl-tabular-data-preparation/data/clean/hitl_cleaned_seed_42.csv'), WindowsPath('C:/Users/ayman/OneDrive/Desktop/hitl-tabular-data-preparation/data/clean/rule_based_cleaned_seed_42.csv'), WindowsPath('C:/Users/ayman/OneDrive/Des

In [72]:
import pandas as pd
import numpy as np

# Load evaluation data
clean_reference = pd.read_csv(
    data_dir / "clean" / "train_clean_reference.csv",
    dtype=str,
    keep_default_na=False
)

ground_truth = pd.read_csv(
    data_dir / "corrupted" / "seed_42" / "ground_truth_mask.csv"
)

hitl_cleaned = pd.read_csv(
    data_dir / "clean" / "hitl_cleaned_seed_42.csv",
    dtype=str,
    keep_default_na=False
)

# All corrupted cells
corrupted_keys = set(
    zip(
        ground_truth["row_index"].astype(int),
        ground_truth["column"]
    )
)

# Cells changed by HITL
changed_keys = set()

for column in hitl_cleaned.columns:
    changed_rows = np.where(
        hitl_cleaned[column].values != clean_reference[column].values
    )[0]

    # We will count actual changes relative to corrupted input below.
    for row_index in changed_rows:
        pass

corrupted_train = pd.read_csv(
    data_dir / "corrupted" / "seed_42" / "corrupted_train.csv",
    dtype=str,
    keep_default_na=False
)

for column in hitl_cleaned.columns:
    changed_rows = np.where(
        hitl_cleaned[column].values != corrupted_train[column].values
    )[0]

    for row_index in changed_rows:
        changed_keys.add((int(row_index), column))

# Correct and incorrect repairs
correct_repairs = 0
incorrect_repairs = 0
harmful_clean_changes = 0

for row_index, column in changed_keys:
    if (row_index, column) in corrupted_keys:
        if (
            hitl_cleaned.at[row_index, column]
            == clean_reference.at[row_index, column]
        ):
            correct_repairs += 1
        else:
            incorrect_repairs += 1
    else:
        harmful_clean_changes += 1

total_corrupted = len(corrupted_keys)
missed_corruptions = total_corrupted - len(
    corrupted_keys.intersection(changed_keys)
)

precision = (
    correct_repairs / (correct_repairs + incorrect_repairs)
    if correct_repairs + incorrect_repairs > 0 else 0
)

recall = (
    correct_repairs / total_corrupted
    if total_corrupted > 0 else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0 else 0
)

hitl_metrics = {
    "seed": 42,
    "total_corrupted": total_corrupted,
    "total_changed": len(changed_keys),
    "correct_repairs": correct_repairs,
    "incorrect_repairs": incorrect_repairs,
    "harmful_clean_changes": harmful_clean_changes,
    "missed_corruptions": missed_corruptions,
    "precision": precision,
    "recall": recall,
    "f1": f1
}

hitl_metrics_df = pd.DataFrame([hitl_metrics])

display(hitl_metrics_df.round(4))

hitl_metrics_df.to_csv(
    results_dir / "cleaning" / "hitl_metrics_seed_42.csv",
    index=False
)

print("Evaluation saved.")

,seed,total_corrupted,total_changed,correct_repairs,incorrect_repairs,harmful_clean_changes,missed_corruptions,precision,recall,f1
0,42,6000,1359,1359,0,0,4641,1.0,0.2265,0.3693


Evaluation saved.


In [73]:
autonomous_df = autonomous_df.drop(
    columns=[":"],
    errors="ignore"
)

autonomous_df = autonomous_df[
    autonomous_df["suggested_value"].notna()
].copy()

print("Autonomous proposals:", len(autonomous_df))

NameError: name 'autonomous_df' is not defined

In [74]:
autonomous_cleaned = pd.read_csv(
    data_dir / "corrupted" / "seed_42" / "corrupted_train.csv",
    dtype=str,
    keep_default_na=False
)

autonomous_logs = []

for _, proposal in autonomous_df.iterrows():
    col = proposal["column"]
    original = str(proposal["original_value"])
    suggested = str(proposal["suggested_value"])

    mask = autonomous_cleaned[col] == original
    count = int(mask.sum())

    autonomous_cleaned.loc[mask, col] = suggested

    autonomous_logs.append({
        "column": col,
        "original_value": original,
        "suggested_value": suggested,
        "affected_cells": count
    })

autonomous_cleaned.to_csv(
    data_dir / "clean" / "autonomous_cleaned_seed_42.csv",
    index=False
)

print("Proposals executed:", len(autonomous_logs))
print("Affected cells:", sum(x["affected_cells"] for x in autonomous_logs))

NameError: name 'autonomous_df' is not defined

In [75]:
def evaluate_agent(original_df, cleaned_df, reference_df, ground_truth_df):

    corrupted_keys = set(zip(
        ground_truth_df["row_index"].astype(int),
        ground_truth_df["column"]
    ))

    changed_keys = set()

    for col in original_df.columns:
        rows = original_df.index[
            original_df[col] != cleaned_df[col]
        ]
        changed_keys.update((int(i), col) for i in rows)

    correct = incorrect = harmful = 0

    for row, col in changed_keys:
        if (row, col) in corrupted_keys:
            if cleaned_df.at[row, col] == reference_df.at[row, col]:
                correct += 1
            else:
                incorrect += 1
        else:
            harmful += 1

    total = len(corrupted_keys)
    missed = total - len(corrupted_keys.intersection(changed_keys))

    precision = correct / (correct + incorrect) if correct + incorrect else 0
    recall = correct / total if total else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

    return pd.DataFrame([{
        "total_corrupted": total,
        "total_changed": len(changed_keys),
        "correct_repairs": correct,
        "incorrect_repairs": incorrect,
        "harmful_clean_changes": harmful,
        "missed_corruptions": missed,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }])

autonomous_metrics = evaluate_agent(
    corrupted_train,
    autonomous_cleaned,
    clean_reference,
    ground_truth
)

display(autonomous_metrics.round(4))

,total_corrupted,total_changed,correct_repairs,incorrect_repairs,harmful_clean_changes,missed_corruptions,precision,recall,f1
0,6000,0,0,0,0,6000,0,0.0,0


In [76]:
comparison = pd.DataFrame([
    {
        "Method": "Rule-Based",
        "Precision": 0.7755,
        "Recall": 0.6690,
        "F1": 0.7183
    },
    {
        "Method": "Autonomous LLM",
        "Precision": 1495 / 1498,
        "Recall": 1495 / 6000,
        "F1": 2 * (1495 / 1498) * (1495 / 6000) /
             ((1495 / 1498) + (1495 / 6000))
    },
    {
        "Method": "HITL LLM",
        "Precision": 1.0,
        "Recall": 1359 / 6000,
        "F1": 2 * 1.0 * (1359 / 6000) /
             (1.0 + (1359 / 6000))
    }
])

display(comparison.round(4))

,Method,Precision,Recall,F1
0,Rule-Based,0.7755,0.6690,0.7183
1,Autonomous LLM,0.9980,0.2492,0.3988
2,HITL LLM,1.0000,0.2265,0.3693


In [77]:
comparison.to_csv(
    results_dir / "cleaning" / "seed_42_method_comparison.csv",
    index=False
)

In [78]:
corrupted_train_123 = pd.read_csv(
    data_dir / "corrupted" / "seed_123" / "corrupted_train.csv",
    dtype=str,
    keep_default_na=False
)

ground_truth_123 = pd.read_csv(
    data_dir / "corrupted" / "seed_123" / "ground_truth_mask.csv"
)

print("Dataset:", corrupted_train_123.shape)
print("Corrupted cells:", len(ground_truth_123))

Dataset: (4000, 16)
Corrupted cells: 6000


In [79]:
result_123 = rule_based_clean(df_corrupted.copy())

print(type(result_123))
print(len(result_123))

<class 'tuple'>
2


In [80]:
df_rule_123, repair_log_123 = result_123

print("Cleaned shape:", df_rule_123.shape)
print("Repair log shape:", repair_log_123.shape)

Cleaned shape: (4000, 16)
Repair log shape: (5172, 5)


In [81]:
output_dir = results_dir / "cleaning"
output_dir.mkdir(parents=True, exist_ok=True)

df_rule_123.to_csv(
    output_dir / "rule_based_cleaned_seed_123.csv",
    index=False
)

repair_log_123.to_csv(
    output_dir / "rule_based_repair_log_seed_123.csv",
    index=False
)

print("Both files saved successfully.")

Both files saved successfully.


In [82]:
clean_reference_123 = pd.read_csv(
    data_dir / "clean" / "train_clean_reference.csv",
    dtype=str,
    keep_default_na=False
)

print(clean_reference_123.shape)

(4000, 16)


In [83]:
def evaluate_rule_results(original_df, cleaned_df, reference_df, mask_df):

    corrupted_keys = set(zip(
        mask_df["row_index"].astype(int),
        mask_df["column"]
    ))

    changed_keys = set()

    for col in original_df.columns:
        original_values = pd.to_numeric(
            original_df[col], errors="ignore"
        ) if col in DOMAIN_RULES else original_df[col].astype(str)

        cleaned_values = (
            pd.to_numeric(cleaned_df[col], errors="ignore")
            if col in DOMAIN_RULES
            else cleaned_df[col].astype(str)
        )

        for i in original_df.index:
            a = original_values.loc[i]
            b = cleaned_values.loc[i]

            if pd.isna(a) and pd.isna(b):
                continue

            if a != b:
                changed_keys.add((int(i), col))

    correct = incorrect = harmful = 0

    for row, col in changed_keys:
        if (row, col) in corrupted_keys:
            if str(cleaned_df.at[row, col]) == str(reference_df.at[row, col]):
                correct += 1
            else:
                incorrect += 1
        else:
            harmful += 1

    total = len(corrupted_keys)
    missed = total - len(corrupted_keys.intersection(changed_keys))

    precision = correct / (correct + incorrect) if correct + incorrect else 0
    recall = correct / total if total else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

    return {
        "seed": 123,
        "total_corrupted": total,
        "total_changed": len(changed_keys),
        "correct_repairs": correct,
        "incorrect_repairs": incorrect,
        "harmful_clean_changes": harmful,
        "missed_corruptions": missed,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

    rule_metrics_123 = evaluate_rule_results(
    df_corrupted,
    df_rule_123,
    clean_reference_123,
    ground_truth
)

print(rule_metrics_123)

NameError: name 'rule_metrics_123' is not defined

In [84]:
for col in ["age", "balance", "day", "campaign", "pdays", "previous"]:
    original = pd.to_numeric(df_corrupted[col], errors="coerce")
    cleaned = pd.to_numeric(df_rule_123[col], errors="coerce")

    differences = (original != cleaned).sum()

    print(f"{col}: {differences} differences")

age: 205 differences
balance: 74 differences
day: 310 differences
campaign: 220 differences
pdays: 209 differences
previous: 226 differences


In [85]:
corrupted_keys = set(zip(
    ground_truth["row_index"].astype(int),
    ground_truth["column"]
))

changed_keys = set()

for col in original_compare.columns:
    changed_rows = original_compare.index[
        original_compare[col] != cleaned_compare[col]
    ]
    changed_keys.update((int(i), col) for i in changed_rows)

correct = 0
incorrect = 0
harmful = 0

for row, col in changed_keys:
    if (row, col) in corrupted_keys:
        if cleaned_compare.at[row, col] == reference_compare.at[row, col]:
            correct += 1
        else:
            incorrect += 1
    else:
        harmful += 1

total = len(corrupted_keys)
missed = total - len(corrupted_keys.intersection(changed_keys))

precision = correct / (correct + incorrect) if correct + incorrect else 0
recall = correct / total
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

rule_metrics_123 = {
    "seed": 123,
    "total_corrupted": total,
    "total_changed": len(changed_keys),
    "correct_repairs": correct,
    "incorrect_repairs": incorrect,
    "harmful_clean_changes": harmful,
    "missed_corruptions": missed,
    "precision": precision,
    "recall": recall,
    "f1": f1
}

display(pd.DataFrame([rule_metrics_123]).round(4))

NameError: name 'original_compare' is not defined

In [86]:
print(rule_metrics_123)

NameError: name 'rule_metrics_123' is not defined

In [87]:
metrics_file = results_dir / "cleaning" / "rule_based_metrics_seed_123.csv"

pd.DataFrame([rule_metrics_123]).to_csv(
    metrics_file,
    index=False
)

print("Saved:", metrics_file)

NameError: name 'rule_metrics_123' is not defined

In [88]:
df_corrupted_2026 = pd.read_csv(
    data_dir / "corrupted" / "seed_2026" / "corrupted_train.csv",
    dtype=str,
    keep_default_na=False
)

ground_truth_2026 = pd.read_csv(
    data_dir / "corrupted" / "seed_2026" / "ground_truth_mask.csv"
)

print("Dataset:", df_corrupted_2026.shape)
print("Corrupted cells:", len(ground_truth_2026))

Dataset: (4000, 16)
Corrupted cells: 6000


In [89]:
result_2026 = rule_based_clean(df_corrupted_2026.copy())

df_rule_2026, repair_log_2026 = result_2026

print("Cleaned shape:", df_rule_2026.shape)
print("Repair log:", repair_log_2026.shape)

Cleaned shape: (4000, 16)
Repair log: (4287, 5)


In [90]:
df_rule_2026.to_csv(
    results_dir / "cleaning" / "rule_based_cleaned_seed_2026.csv",
    index=False
)

repair_log_2026.to_csv(
    results_dir / "cleaning" / "rule_based_repair_log_seed_2026.csv",
    index=False
)

print("Seed 2026 files saved successfully.")

Seed 2026 files saved successfully.


In [91]:
clean_reference_2026 = pd.read_csv(
    data_dir / "clean" / "train_clean_reference.csv",
    dtype=str,
    keep_default_na=False
)

print("Reference shape:", clean_reference_2026.shape)

Reference shape: (4000, 16)


In [92]:
numeric_cols = [
    "age", "balance", "day",
    "campaign", "pdays", "previous"
]

original_compare_2026 = df_corrupted_2026.copy()
cleaned_compare_2026 = df_rule_2026.copy()
reference_compare_2026 = clean_reference_2026.copy()

for col in numeric_cols:
    original_compare_2026[col] = pd.to_numeric(
        original_compare_2026[col], errors="coerce"
    )
    cleaned_compare_2026[col] = pd.to_numeric(
        cleaned_compare_2026[col], errors="coerce"
    )
    reference_compare_2026[col] = pd.to_numeric(
        reference_compare_2026[col], errors="coerce"
    )

print("Comparison copies prepared.")

Comparison copies prepared.


In [93]:
corrupted_keys_2026 = set(zip(
    ground_truth_2026["row_index"].astype(int),
    ground_truth_2026["column"]
))

changed_keys_2026 = set()

for col in original_compare_2026.columns:
    changed_rows = original_compare_2026.index[
        original_compare_2026[col] != cleaned_compare_2026[col]
    ]
    changed_keys_2026.update((int(i), col) for i in changed_rows)

correct_2026 = 0
incorrect_2026 = 0
harmful_2026 = 0

for row, col in changed_keys_2026:
    if (row, col) in corrupted_keys_2026:
        if cleaned_compare_2026.at[row, col] == reference_compare_2026.at[row, col]:
            correct_2026 += 1
        else:
            incorrect_2026 += 1
    else:
        harmful_2026 += 1

total_2026 = len(corrupted_keys_2026)
missed_2026 = total_2026 - len(
    corrupted_keys_2026.intersection(changed_keys_2026)
)

precision_2026 = (
    correct_2026 / (correct_2026 + incorrect_2026)
    if correct_2026 + incorrect_2026 else 0
)

recall_2026 = correct_2026 / total_2026

f1_2026 = (
    2 * precision_2026 * recall_2026 /
    (precision_2026 + recall_2026)
    if precision_2026 + recall_2026 else 0
)

rule_metrics_2026 = {
    "seed": 2026,
    "total_corrupted": total_2026,
    "total_changed": len(changed_keys_2026),
    "correct_repairs": correct_2026,
    "incorrect_repairs": incorrect_2026,
    "harmful_clean_changes": harmful_2026,
    "missed_corruptions": missed_2026,
    "precision": precision_2026,
    "recall": recall_2026,
    "f1": f1_2026
}

display(pd.DataFrame([rule_metrics_2026]).round(4))

,seed,total_corrupted,total_changed,correct_repairs,incorrect_repairs,harmful_clean_changes,missed_corruptions,precision,recall,f1
0,2026,6000,4287,3476,811,0,1713,0.8108,0.5793,0.6758


In [94]:
print(rule_metrics_2026)

{'seed': 2026, 'total_corrupted': 6000, 'total_changed': 4287, 'correct_repairs': 3476, 'incorrect_repairs': 811, 'harmful_clean_changes': 0, 'missed_corruptions': 1713, 'precision': 0.8108234196407744, 'recall': 0.5793333333333334, 'f1': 0.6758044133372219}


In [95]:
pd.DataFrame([rule_metrics_2026]).to_csv(
    results_dir / "cleaning" / "rule_based_metrics_seed_2026.csv",
    index=False
)

print("Seed 2026 metrics saved successfully.")

Seed 2026 metrics saved successfully.


In [96]:
metrics_dir = results_dir / "cleaning"

for file in sorted(metrics_dir.glob("*metrics*seed*.csv")):
    print(file.name)

hitl_metrics_seed_42.csv
rule_based_metrics_seed_123.csv
rule_based_metrics_seed_2026.csv
rule_based_metrics_seed_42.csv


In [97]:
for file in sorted(metrics_dir.glob("*42*.csv")):
    print(file.name)

hitl_metrics_seed_42.csv
rule_based_metrics_seed_42.csv
seed_42_method_comparison.csv
suspicious_values_seed_42.csv


In [98]:
comparison_42 = pd.read_csv(
    metrics_dir / "seed_42_method_comparison.csv"
)

display(comparison_42)

,Method,Precision,Recall,F1
0,Rule-Based,0.775500,0.669000,0.718300
1,Autonomous LLM,0.997997,0.249167,0.398773
2,HITL LLM,1.000000,0.226500,0.369344


In [99]:
variables_to_check = [
    "df_corrupted",
    "df_rule_cleaned",
    "clean_reference",
    "ground_truth",
    "df_corrupted_2026",
    "df_rule_2026"
]

for name in variables_to_check:
    print(f"{name}: {'Available' if name in globals() else 'Not available'}")

df_corrupted: Available
df_rule_cleaned: Available
clean_reference: Available
ground_truth: Available
df_corrupted_2026: Available
df_rule_2026: Available


In [100]:
numeric_cols = [
    "age", "balance", "day",
    "campaign", "pdays", "previous"
]

original_compare_42 = df_corrupted.copy()
cleaned_compare_42 = df_rule_cleaned.copy()
reference_compare_42 = clean_reference.copy()

for col in numeric_cols:
    original_compare_42[col] = pd.to_numeric(
        original_compare_42[col], errors="coerce"
    )
    cleaned_compare_42[col] = pd.to_numeric(
        cleaned_compare_42[col], errors="coerce"
    )
    reference_compare_42[col] = pd.to_numeric(
        reference_compare_42[col], errors="coerce"
    )

print("Seed 42 comparison copies prepared.")

Seed 42 comparison copies prepared.


In [101]:
corrupted_keys_42 = set(zip(
    ground_truth["row_index"].astype(int),
    ground_truth["column"]
))

changed_keys_42 = set()

for col in original_compare_42.columns:
    changed_rows = original_compare_42.index[
        original_compare_42[col] != cleaned_compare_42[col]
    ]
    changed_keys_42.update((int(i), col) for i in changed_rows)

correct_42 = 0
incorrect_42 = 0
harmful_42 = 0

for row, col in changed_keys_42:
    if (row, col) in corrupted_keys_42:
        if cleaned_compare_42.at[row, col] == reference_compare_42.at[row, col]:
            correct_42 += 1
        else:
            incorrect_42 += 1
    else:
        harmful_42 += 1

total_42 = len(corrupted_keys_42)
missed_42 = total_42 - len(corrupted_keys_42.intersection(changed_keys_42))

precision_42 = correct_42 / (correct_42 + incorrect_42) if correct_42 + incorrect_42 else 0
recall_42 = correct_42 / total_42
f1_42 = 2 * precision_42 * recall_42 / (precision_42 + recall_42) if precision_42 + recall_42 else 0

rule_metrics_42 = {
    "seed": 42,
    "total_corrupted": total_42,
    "total_changed": len(changed_keys_42),
    "correct_repairs": correct_42,
    "incorrect_repairs": incorrect_42,
    "harmful_clean_changes": harmful_42,
    "missed_corruptions": missed_42,
    "precision": precision_42,
    "recall": recall_42,
    "f1": f1_42
}

print(rule_metrics_42)

{'seed': 42, 'total_corrupted': 6000, 'total_changed': 5172, 'correct_repairs': 4004, 'incorrect_repairs': 1168, 'harmful_clean_changes': 0, 'missed_corruptions': 828, 'precision': 0.774168600154679, 'recall': 0.6673333333333333, 'f1': 0.7167919799498748}


In [102]:
print("Dataset shapes:")
print("Corrupted:", df_corrupted.shape)
print("Cleaned:", df_rule_cleaned.shape)
print("Reference:", clean_reference.shape)
print("Mask:", len(ground_truth))

print("\nChanges by column:")
display(
    (original_compare_42 != cleaned_compare_42).sum()
)

Dataset shapes:
Corrupted: (4000, 16)
Cleaned: (4000, 16)
Reference: (4000, 16)
Mask: 6000

Changes by column:


age          205
job          442
marital      431
education    430
default      441
balance       74
housing      479
loan         449
contact      407
day          310
month        438
campaign     220
pdays        209
previous     226
poutcome     411
y              0
dtype: int64

In [103]:
print("Repair log variables:")

for name in ["repair_log", "df_rule_123", "repair_log_123"]:
    if name in globals():
        obj = globals()[name]
        print(name, obj.shape if hasattr(obj, "shape") else type(obj))

print("\nSaved Seed 42 files:")

for file in sorted(results_dir.rglob("*42*")):
    print(file.name)

Repair log variables:
repair_log (5172, 5)
df_rule_123 (4000, 16)
repair_log_123 (5172, 5)

Saved Seed 42 files:
hitl_metrics_seed_42.csv
rule_based_metrics_seed_42.csv
seed_42
seed_42_method_comparison.csv
suspicious_values_seed_42.csv
column_profiles_seed_42.csv
column_profiles_seed_42.json
hitl_decisions_seed_42.csv
hitl_execution_log_seed_42.json
hitl_pilot_decisions_seed_42_contact.csv
hitl_pilot_execution_log_seed_42_contact.csv
hitl_pilot_metrics_seed_42_contact.csv
hitl_pilot_review_time_seed_42.json
hitl_review_seed_42.csv
hitl_review_summary_seed_42.json
llm_all_proposals_seed_42.csv
llm_operational_metrics_seed_42.csv


In [104]:
df_corrupted_42 = pd.read_csv(
    data_dir / "corrupted" / "seed_42" / "corrupted_train.csv",
    dtype=str,
    keep_default_na=False,
    na_values=[""]
)

df_rule_42_new, repair_log_42_new = rule_based_clean(
    df_corrupted_42.copy()
)

print("Repair log:", len(repair_log_42_new))
print(repair_log_42_new["operation"].value_counts())

Repair log: 5172
operation
fuzzy_matching                     1500
format_normalization               1500
categorical_mode_imputation         928
invalid_value_median_imputation     672
numerical_median_imputation         572
Name: count, dtype: int64


In [105]:
numeric_cols = [
    "age", "balance", "day",
    "campaign", "pdays", "previous"
]

original_compare_42 = df_corrupted_42.copy()
cleaned_compare_42 = df_rule_42_new.copy()
reference_compare_42 = clean_reference_42.copy()

for col in numeric_cols:
    original_compare_42[col] = pd.to_numeric(
        original_compare_42[col], errors="coerce"
    )
    cleaned_compare_42[col] = pd.to_numeric(
        cleaned_compare_42[col], errors="coerce"
    )
    reference_compare_42[col] = pd.to_numeric(
        reference_compare_42[col], errors="coerce"
    )

print("Seed 42 comparison copies prepared.")

NameError: name 'clean_reference_42' is not defined

In [ ]:
corrupted_keys_42 = set(zip(
    ground_truth_42["row_index"].astype(int),
    ground_truth_42["column"]
))

changed_keys_42 = set()

for col in original_compare_42.columns:
    changed_rows = original_compare_42.index[
        original_compare_42[col] != cleaned_compare_42[col]
    ]
    changed_keys_42.update((int(i), col) for i in changed_rows)

correct_42 = incorrect_42 = harmful_42 = 0

for row, col in changed_keys_42:
    if (row, col) in corrupted_keys_42:
        if cleaned_compare_42.at[row, col] == reference_compare_42.at[row, col]:
            correct_42 += 1
        else:
            incorrect_42 += 1
    else:
        harmful_42 += 1

total_42 = len(corrupted_keys_42)
missed_42 = total_42 - len(corrupted_keys_42.intersection(changed_keys_42))

precision_42 = correct_42 / (correct_42 + incorrect_42) if correct_42 + incorrect_42 else 0
recall_42 = correct_42 / total_42
f1_42 = 2 * precision_42 * recall_42 / (precision_42 + recall_42) if precision_42 + recall_42 else 0

rule_metrics_42 = {
    "seed": 42,
    "total_corrupted": total_42,
    "total_changed": len(changed_keys_42),
    "correct_repairs": correct_42,
    "incorrect_repairs": incorrect_42,
    "harmful_clean_changes": harmful_42,
    "missed_corruptions": missed_42,
    "precision": precision_42,
    "recall": recall_42,
    "f1": f1_42
}

print(rule_metrics_42)

In [ ]:
pd.DataFrame([rule_metrics_42]).to_csv(
    results_dir / "cleaning" / "rule_based_metrics_seed_42.csv",
    index=False
)

print("Seed 42 metrics saved successfully.")

In [106]:
metrics_files = [
    metrics_dir / "rule_based_metrics_seed_42.csv",
    metrics_dir / "rule_based_metrics_seed_123.csv",
    metrics_dir / "rule_based_metrics_seed_2026.csv"
]

all_rule_results = pd.concat(
    [pd.read_csv(file) for file in metrics_files],
    ignore_index=True
)

display(all_rule_results.round(4))

,seed,total_corrupted,total_changed,correct_repairs,incorrect_repairs,harmful_clean_changes,missed_corruptions,precision,recall,f1
0,42,6000,5172,4004,1168,0,828,0.7742,0.6673,0.7168
1,123,6000,4234,3448,786,0,1766,0.8144,0.5747,0.6738
2,2026,6000,4287,3476,811,0,1713,0.8108,0.5793,0.6758


In [107]:
rule_based_summary = all_rule_results[
    ["precision", "recall", "f1"]
].agg(["mean", "std"]).T

rule_based_summary.columns = ["mean", "std"]

display(rule_based_summary.round(4))

rule_based_summary.to_csv(
    metrics_dir / "rule_based_summary_statistics.csv"
)

print("Rule-Based summary statistics saved successfully.")

,mean,std
precision,0.7998,0.0223
recall,0.6071,0.0522
f1,0.6888,0.0243


Rule-Based summary statistics saved successfully.


In [108]:
from pathlib import Path

for seed in [42, 123, 2026]:
    folder = results_dir / "cleaning"
    files = list(folder.glob(f"*autonomous*{seed}*"))
    print(f"\nSeed {seed}:")
    for file in files:
        print(file.name)


Seed 42:

Seed 123:

Seed 2026:


In [109]:
for seed in [42, 123, 2026]:
    folder = data_dir / "clean"
    files = list(folder.glob(f"*{seed}*"))
    print(f"\nSeed {seed}:")
    for file in files:
        print(file.name)


Seed 42:
autonomous_cleaned_seed_42.csv
hitl_cleaned_seed_42.csv
rule_based_cleaned_seed_42.csv

Seed 123:
autonomous_cleaned_seed_123.csv
hitl_cleaned_seed_123.csv

Seed 2026:
autonomous_cleaned_seed_2026.csv
hitl_cleaned_seed_2026.csv


In [110]:
df_corrupted_123 = pd.read_csv(
    data_dir / "corrupted" / "seed_123" / "corrupted_train.csv",
    dtype=str,
    keep_default_na=False
)

ground_truth_123 = pd.read_csv(
    data_dir / "corrupted" / "seed_123" / "ground_truth_mask.csv"
)

print(df_corrupted_123.shape)
print(ground_truth_123.shape)

(4000, 16)
(6000, 5)


In [111]:
suspicious_values_123 = {}

for col in VALID_CATEGORIES:
    values = (
        df_corrupted_123[col]
        .dropna()
        .astype(str)
        .unique()
    )
    suspicious_values_123[col] = sorted(
        v for v in values
        if v.strip().lower() not in VALID_CATEGORIES[col]
    )

for col, values in suspicious_values_123.items():
    if values:
        print(col, values)

job ['', 'adin.', 'admi.', 'admin', 'admn.', 'amin.', 'anagement', 'ble-collar', 'blu-collar', 'blue-cllar', 'blue-colar', 'blue-colla', 'blue-collr', 'blue-ollar', 'bluecollar', 'bue-collar', 'dmin.', 'echnician', 'enrepreneur', 'entepreneur', 'entrepreneu', 'entreprenur', 'entreprneur', 'ervices', 'etired', 'etrepreneur', 'houemaid', 'houseaid', 'housemai', 'housemid', 'housmaid', 'lue-collar', 'maagement', 'manaement', 'manageent', 'managemen', 'managemet', 'managemnt', 'managment', 'mangement', 'mnagement', 'ousemaid', 'reired', 'retied', 'retred', 'rtired', 'self-emloyed', 'serices', 'servces', 'servics', 'servies', 'sevices', 'srvices', 'studen', 'studet', 'studnt', 'stuent', 'sudent', 'tchnician', 'techician', 'techncian', 'technican', 'technicia', 'technicin', 'techniian', 'tecnician', 'tehnician', 'uknown', 'unemploed', 'unemploye', 'unempoyed']
marital ['', 'arried', 'diorced', 'divoced', 'divorcd', 'divorce', 'divored', 'divrced', 'dvorced', 'ingle', 'ivorced', 'maried', 'ma

In [112]:
import ast

tree = ast.parse(planner_path.read_text(encoding="utf-8"))

for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
        print(node.name)

NameError: name 'planner_path' is not defined

In [113]:
import inspect

print(inspect.signature(propose_corrections))
print(inspect.getsource(propose_corrections))

(suspicious_values, allowed_values)
def propose_corrections(suspicious_values, allowed_values):

    prompt = f"""
You are a data cleaning assistant.

Suggest corrections for suspicious categorical values.

Rules:
1. Only suggest values from the allowed values list.
2. Never invent new categories.
3. Preserve valid meanings such as "unknown".
4. If uncertain, use null.
5. Do not modify any dataset.
6. Return valid JSON only.

Allowed values:
{json.dumps(allowed_values, ensure_ascii=False)}

Suspicious values and frequencies:
{json.dumps(suspicious_values, ensure_ascii=False)}

Return this JSON structure:
{{
    "proposals": [
        {{
            "original_value": "example",
            "suggested_value": "example",
            "reason": "short explanation",
            "confidence": 0.0
        }}
    ]
}}
"""

    start = time.perf_counter()

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "messages": [
                {

In [114]:
for name in globals():
    if any(word in name.lower() for word in ["proposal", "batch", "operational"]):
        print(name, type(globals()[name]).__name__)

proposal_validator module
validate_proposals function
valid_proposals list
rejected_proposals list
execute_categorical_proposals function
all_llm_proposals dict
column_proposals list
batch dict
proposals list
proposal_rows list
proposal dict
proposals_df DataFrame
batches list
show_proposal function


In [115]:
for i, cell in enumerate(In):
    if "propose_corrections(" in cell and "def propose_corrections" not in cell:
        print(f"\n--- Cell {i} ---\n")
        print(cell[:3000])


--- Cell 49 ---

test_column = "contact"

test_values = (
    suspicious_df[
        suspicious_df["column"] == test_column
    ]
    .head(20)
    .to_dict(orient="records")
)

test_result = propose_corrections(
    suspicious_values=test_values,
    allowed_values=ALLOWED_VALUES[test_column]
)

print(json.dumps(
    test_result,
    indent=2,
    ensure_ascii=False
))

--- Cell 50 ---

test_column = "contact"

test_values_5 = (
    suspicious_df[
        suspicious_df["column"] == test_column
    ]
    .head(5)
    .to_dict(orient="records")
)

test_result_5 = propose_corrections(
    suspicious_values=test_values_5,
    allowed_values=ALLOWED_VALUES[test_column]
)

print(json.dumps(
    test_result_5,
    indent=2,
    ensure_ascii=False
))

--- Cell 63 ---

all_llm_proposals = {}
all_llm_metrics = {}

for column, values in all_suspicious_values.items():
    print(f"\nProcessing: {column} ({len(values)} values)")

    allowed = ALLOWED_VALUES[column]

    column_proposals = []
    

In [116]:
print("Suspicious DF:")
display(suspicious_df.head())

print("\nColumns:", suspicious_df.columns.tolist())
print("\nAllowed values available:", list(ALLOWED_VALUES.keys()))

Suspicious DF:


,column,suspicious_value,frequency
116,contact,celular,25
117,contact,ellular,19
118,contact,cllular,18
119,contact,celluar,12
120,contact,unkown,11



Columns: ['column', 'suspicious_value', 'frequency']

Allowed values available: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [117]:
suspicious_rows = []

for col, allowed in ALLOWED_VALUES.items():
    counts = df_corrupted_123[col].value_counts()

    for value, frequency in counts.items():
        if value.strip().lower() not in allowed and value.strip() != "":
            suspicious_rows.append({
                "column": col,
                "suspicious_value": value,
                "frequency": frequency
            })

suspicious_df_123 = pd.DataFrame(suspicious_rows)

print("Unique suspicious values:", len(suspicious_df_123))
print("Total suspicious cells:", suspicious_df_123["frequency"].sum())

display(suspicious_df_123.head(10))

Unique suspicious values: 173
Total suspicious cells: 1500


,column,suspicious_value,frequency
0,job,mnagement,7
1,job,dmin.,7
2,job,blue-colar,6
3,job,tchnician,6
4,job,servics,5
5,job,admn.,5
6,job,tecnician,5
7,job,techician,5
8,job,mangement,5
9,job,admi.,5


In [118]:
all_llm_proposals_123 = {}
all_llm_metrics_123 = {}

for column in ALLOWED_VALUES:
    values = (
        suspicious_df_123[
            suspicious_df_123["column"] == column
        ]
        .to_dict(orient="records")
    )

    if not values:
        continue

    print(f"Processing {column}: {len(values)} values")

    column_proposals = []
    column_metrics = []

    for start in range(0, len(values), 5):
        batch_values = values[start:start + 5]

        result = propose_corrections(
            batch_values,
            ALLOWED_VALUES[column]
        )

        proposals = result["response"].get("proposals", [])

        valid, rejected = validate_proposals(
            proposals,
            batch_values,
            ALLOWED_VALUES[column]
        )

        column_proposals.extend(valid)

        column_metrics.append({
            "column": column,
            "batch": start // 5 + 1,
            "input_values": len(batch_values),
            "returned": len(proposals),
            "valid": len(valid),
            "rejected": len(rejected),
            "runtime_seconds": result["runtime_seconds"],
            "prompt_tokens": result["prompt_tokens"],
            "completion_tokens": result["completion_tokens"]
        })

    all_llm_proposals_123[column] = column_proposals
    all_llm_metrics_123[column] = column_metrics

print("\nSeed 123 proposals completed.")

Processing job: 70 values
Processing marital: 20 values
Processing education: 28 values
Processing default: 2 values
Processing housing: 2 values
Processing loan: 2 values
Processing contact: 21 values
Processing month: 10 values
Processing poutcome: 18 values

Seed 123 proposals completed.


In [119]:
total_valid_123 = sum(
    len(v) for v in all_llm_proposals_123.values()
)

total_batches_123 = sum(
    len(v) for v in all_llm_metrics_123.values()
)

total_rejected_123 = sum(
    item["rejected"]
    for batches_list in all_llm_metrics_123.values()
    for item in batches_list
)

print("Valid proposals:", total_valid_123)
print("Rejected proposals:", total_rejected_123)
print("Total batches:", total_batches_123)
print("Total runtime (seconds):",
      round(sum(
          item["runtime_seconds"]
          for batches_list in all_llm_metrics_123.values()
          for item in batches_list
      ), 2))

Valid proposals: 173
Rejected proposals: 0
Total batches: 38
Total runtime (seconds): 584.63


In [120]:
from pathlib import Path

project_root = Path.cwd().parent

operational_dir = project_root / "results" / "operational"
operational_dir.mkdir(parents=True, exist_ok=True)

proposals_df_123.to_csv(
    operational_dir / "llm_all_proposals_seed_123.csv",
    index=False
)

metrics_df_123.to_csv(
    operational_dir / "llm_operational_metrics_seed_123.csv",
    index=False
)

print("Seed 123 files saved successfully.")

NameError: name 'proposals_df_123' is not defined

In [ ]:
proposals_rows_123 = [
    proposal
    for column_proposals in all_llm_proposals_123.values()
    for proposal in column_proposals
]

proposals_df_123 = pd.DataFrame(proposals_rows_123)

metrics_rows_123 = [
    item
    for column_batches in all_llm_metrics_123.values()
    for item in column_batches
]

metrics_df_123 = pd.DataFrame(metrics_rows_123)

proposals_df_123.to_csv(
    project_root / "results" / "operational" / "llm_all_proposals_seed_123.csv",
    index=False
)

metrics_df_123.to_csv(
    project_root / "results" / "operational" / "llm_operational_metrics_seed_123.csv",
    index=False
)

print("Seed 123 proposals and operational metrics saved.")

In [ ]:
import inspect

print(inspect.signature(execute_categorical_proposals))
print(inspect.getsource(execute_categorical_proposals))

In [ ]:
autonomous_cleaned_123 = df_corrupted_123.copy()
autonomous_log_123 = []

for column in all_llm_proposals_123:
    autonomous_cleaned_123, log = execute_categorical_proposals(
        dataframe=autonomous_cleaned_123,
        column=column,
        valid_proposals=all_llm_proposals_123[column]
    )
    autonomous_log_123.extend(log)

clean_dir = project_root / "data" / "clean"

autonomous_cleaned_123.to_csv(
    clean_dir / "autonomous_cleaned_seed_123.csv",
    index=False
)

pd.DataFrame(autonomous_log_123).to_csv(
    operational_dir / "autonomous_execution_log_seed_123.csv",
    index=False
)

print("Executed proposals:", len(autonomous_log_123))
print("Affected cells:", sum(x["affected_rows"] for x in autonomous_log_123))
print("Autonomous Seed 123 saved successfully.")

In [ ]:
clean_reference_123 = pd.read_csv(
    project_root / "data" / "clean" / "train_clean_reference.csv",
    dtype=str,
    keep_default_na=False
)

numeric_cols = ["age", "balance", "day", "campaign", "pdays", "previous"]

original_compare_123 = df_corrupted_123.copy()
cleaned_compare_123 = autonomous_cleaned_123.copy()
reference_compare_123 = clean_reference_123.copy()

for col in numeric_cols:
    original_compare_123[col] = pd.to_numeric(original_compare_123[col], errors="coerce")
    cleaned_compare_123[col] = pd.to_numeric(cleaned_compare_123[col], errors="coerce")
    reference_compare_123[col] = pd.to_numeric(reference_compare_123[col], errors="coerce")

corrupted_keys_123 = set(zip(
    ground_truth_123["row_index"].astype(int),
    ground_truth_123["column"]
))

changed_keys_123 = set()

for col in original_compare_123.columns:
    changed_rows = original_compare_123.index[
        original_compare_123[col] != cleaned_compare_123[col]
    ]
    changed_keys_123.update((int(i), col) for i in changed_rows)

correct_123 = incorrect_123 = harmful_123 = 0

for row, col in changed_keys_123:
    if (row, col) in corrupted_keys_123:
        if cleaned_compare_123.at[row, col] == reference_compare_123.at[row, col]:
            correct_123 += 1
        else:
            incorrect_123 += 1
    else:
        harmful_123 += 1

total_123 = len(corrupted_keys_123)
missed_123 = total_123 - len(corrupted_keys_123.intersection(changed_keys_123))

precision_123 = correct_123 / (correct_123 + incorrect_123)
recall_123 = correct_123 / total_123
f1_123 = 2 * precision_123 * recall_123 / (precision_123 + recall_123)

autonomous_metrics_123 = {
    "seed": 123,
    "total_corrupted": total_123,
    "total_changed": len(changed_keys_123),
    "correct_repairs": correct_123,
    "incorrect_repairs": incorrect_123,
    "harmful_clean_changes": harmful_123,
    "missed_corruptions": missed_123,
    "precision": precision_123,
    "recall": recall_123,
    "f1": f1_123
}

print(autonomous_metrics_123)

In [ ]:
changed_keys_123 = set()

for col in original_compare_123.columns:
    original = original_compare_123[col]
    cleaned = cleaned_compare_123[col]

    changed_mask = (
        original.ne(cleaned).fillna(False)
        & ~(original.isna() & cleaned.isna())
    )

    changed_rows = original_compare_123.index[changed_mask]

    changed_keys_123.update(
        (int(i), col) for i in changed_rows
    )

correct_123 = incorrect_123 = harmful_123 = 0

for row, col in changed_keys_123:
    if (row, col) in corrupted_keys_123:
        if cleaned_compare_123.at[row, col] == reference_compare_123.at[row, col]:
            correct_123 += 1
        else:
            incorrect_123 += 1
    else:
        harmful_123 += 1

total_123 = len(corrupted_keys_123)
missed_123 = total_123 - len(corrupted_keys_123.intersection(changed_keys_123))

precision_123 = correct_123 / (correct_123 + incorrect_123)
recall_123 = correct_123 / total_123
f1_123 = 2 * precision_123 * recall_123 / (precision_123 + recall_123)

autonomous_metrics_123 = {
    "seed": 123,
    "total_corrupted": total_123,
    "total_changed": len(changed_keys_123),
    "correct_repairs": correct_123,
    "incorrect_repairs": incorrect_123,
    "harmful_clean_changes": harmful_123,
    "missed_corruptions": missed_123,
    "precision": precision_123,
    "recall": recall_123,
    "f1": f1_123
}

print(autonomous_metrics_123)

In [ ]:
pd.DataFrame([autonomous_metrics_123]).to_csv(
    operational_dir / "autonomous_metrics_seed_123.csv",
    index=False
)

print("Autonomous Seed 123 metrics saved.")

In [ ]:
df_corrupted_2026 = pd.read_csv(
    project_root / "data" / "corrupted" / "seed_2026" / "corrupted_train.csv",
    dtype=str,
    keep_default_na=False
)

ground_truth_2026 = pd.read_csv(
    project_root / "data" / "corrupted" / "seed_2026" / "ground_truth_mask.csv"
)

print(df_corrupted_2026.shape)
print(ground_truth_2026.shape)

In [ ]:
suspicious_rows_2026 = []

for col, allowed in ALLOWED_VALUES.items():
    counts = df_corrupted_2026[col].value_counts()

    for value, frequency in counts.items():
        if value.strip().lower() not in allowed and value.strip() != "":
            suspicious_rows_2026.append({
                "column": col,
                "suspicious_value": value,
                "frequency": frequency
            })

suspicious_df_2026 = pd.DataFrame(suspicious_rows_2026)

print("Unique suspicious values:", len(suspicious_df_2026))
print("Total suspicious cells:", suspicious_df_2026["frequency"].sum())

In [ ]:
all_llm_proposals_2026 = {}
all_llm_metrics_2026 = {}

for column in ALLOWED_VALUES:
    values = (
        suspicious_df_2026[
            suspicious_df_2026["column"] == column
        ].to_dict(orient="records")
    )

    if not values:
        continue

    print(f"Processing {column}: {len(values)} values")

    column_proposals = []
    column_metrics = []

    for start in range(0, len(values), 5):
        batch_values = values[start:start + 5]

        result = propose_corrections(
            batch_values,
            ALLOWED_VALUES[column]
        )

        proposals = result["response"].get("proposals", [])

        valid, rejected = validate_proposals(
            proposals,
            batch_values,
            ALLOWED_VALUES[column]
        )

        column_proposals.extend(valid)

        column_metrics.append({
            "column": column,
            "batch": start // 5 + 1,
            "input_values": len(batch_values),
            "returned": len(proposals),
            "valid": len(valid),
            "rejected": len(rejected),
            "runtime_seconds": result["runtime_seconds"],
            "prompt_tokens": result["prompt_tokens"],
            "completion_tokens": result["completion_tokens"]
        })

    all_llm_proposals_2026[column] = column_proposals
    all_llm_metrics_2026[column] = column_metrics

print("\nSeed 2026 proposals completed.")

In [ ]:
total_valid_2026 = sum(len(v) for v in all_llm_proposals_2026.values())

total_batches_2026 = sum(
    len(v) for v in all_llm_metrics_2026.values()
)

total_rejected_2026 = sum(
    item["rejected"]
    for items in all_llm_metrics_2026.values()
    for item in items
)

total_runtime_2026 = sum(
    item["runtime_seconds"]
    for items in all_llm_metrics_2026.values()
    for item in items
)

print("Valid proposals:", total_valid_2026)
print("Rejected proposals:", total_rejected_2026)
print("Total batches:", total_batches_2026)
print("Runtime (seconds):", round(total_runtime_2026, 2))

In [ ]:
proposals_rows_2026 = [
    p for items in all_llm_proposals_2026.values()
    for p in items
]

metrics_rows_2026 = [
    item for items in all_llm_metrics_2026.values()
    for item in items
]

proposals_df_2026 = pd.DataFrame(proposals_rows_2026)
metrics_df_2026 = pd.DataFrame(metrics_rows_2026)

proposals_df_2026.to_csv(
    operational_dir / "llm_all_proposals_seed_2026.csv",
    index=False
)

metrics_df_2026.to_csv(
    operational_dir / "llm_operational_metrics_seed_2026.csv",
    index=False
)

print("Seed 2026 proposals and operational metrics saved.")

In [ ]:
autonomous_cleaned_2026 = df_corrupted_2026.copy()
autonomous_log_2026 = []

for column in all_llm_proposals_2026:
    autonomous_cleaned_2026, log = execute_categorical_proposals(
        dataframe=autonomous_cleaned_2026,
        column=column,
        valid_proposals=all_llm_proposals_2026[column]
    )
    autonomous_log_2026.extend(log)

autonomous_cleaned_2026.to_csv(
    project_root / "data" / "clean" / "autonomous_cleaned_seed_2026.csv",
    index=False
)

pd.DataFrame(autonomous_log_2026).to_csv(
    operational_dir / "autonomous_execution_log_seed_2026.csv",
    index=False
)

print("Executed proposals:", len(autonomous_log_2026))
print("Affected cells:", sum(x["affected_rows"] for x in autonomous_log_2026))
print("Autonomous Seed 2026 saved successfully.")

In [ ]:
clean_reference_2026 = pd.read_csv(
    project_root / "data" / "clean" / "train_clean_reference.csv",
    dtype=str,
    keep_default_na=False
)

numeric_cols = ["age", "balance", "day", "campaign", "pdays", "previous"]

original_compare_2026 = df_corrupted_2026.copy()
cleaned_compare_2026 = autonomous_cleaned_2026.copy()
reference_compare_2026 = clean_reference_2026.copy()

for col in numeric_cols:
    original_compare_2026[col] = pd.to_numeric(original_compare_2026[col], errors="coerce")
    cleaned_compare_2026[col] = pd.to_numeric(cleaned_compare_2026[col], errors="coerce")
    reference_compare_2026[col] = pd.to_numeric(reference_compare_2026[col], errors="coerce")

corrupted_keys_2026 = set(zip(
    ground_truth_2026["row_index"].astype(int),
    ground_truth_2026["column"]
))

changed_keys_2026 = set()

for col in original_compare_2026.columns:
    original = original_compare_2026[col]
    cleaned = cleaned_compare_2026[col]

    changed_mask = (
        original.ne(cleaned).fillna(False)
        & ~(original.isna() & cleaned.isna())
    )

    changed_rows = original_compare_2026.index[changed_mask]
    changed_keys_2026.update((int(i), col) for i in changed_rows)

correct_2026 = incorrect_2026 = harmful_2026 = 0

for row, col in changed_keys_2026:
    if (row, col) in corrupted_keys_2026:
        if cleaned_compare_2026.at[row, col] == reference_compare_2026.at[row, col]:
            correct_2026 += 1
        else:
            incorrect_2026 += 1
    else:
        harmful_2026 += 1

total_2026 = len(corrupted_keys_2026)
missed_2026 = total_2026 - len(corrupted_keys_2026.intersection(changed_keys_2026))

precision_2026 = correct_2026 / (correct_2026 + incorrect_2026)
recall_2026 = correct_2026 / total_2026
f1_2026 = 2 * precision_2026 * recall_2026 / (precision_2026 + recall_2026)

autonomous_metrics_2026 = {
    "seed": 2026,
    "total_corrupted": total_2026,
    "total_changed": len(changed_keys_2026),
    "correct_repairs": correct_2026,
    "incorrect_repairs": incorrect_2026,
    "harmful_clean_changes": harmful_2026,
    "missed_corruptions": missed_2026,
    "precision": precision_2026,
    "recall": recall_2026,
    "f1": f1_2026
}

print(autonomous_metrics_2026)

In [ ]:
pd.DataFrame([autonomous_metrics_2026]).to_csv(
    operational_dir / "autonomous_metrics_seed_2026.csv",
    index=False
)

print("Autonomous Seed 2026 metrics saved.")

In [ ]:
import inspect

print(inspect.signature(show_proposal))
print(inspect.getsource(show_proposal))

In [ ]:
print("Approve callbacks:")
print(approve_btn._click_handlers.callbacks)

print("\nReject callbacks:")
print(reject_btn._click_handlers.callbacks)

In [ ]:
import dis

print("APPROVE:")
dis.dis(approve_btn._click_handlers.callbacks[0])

print("\nREJECT:")
dis.dis(reject_btn._click_handlers.callbacks[0])

In [ ]:
print(inspect.getsource(record_decision))

In [ ]:
review_df_123 = pd.read_csv(
    operational_dir / "llm_all_proposals_seed_123.csv"
)

review_df_123 = review_df_123.reset_index(drop=True)

print("Total proposals:", len(review_df_123))
print(review_df_123.head())
print(review_df_123.columns.tolist())

In [ ]:
review_rows_123 = []

for column, proposals in all_llm_proposals_123.items():
    for proposal in proposals:
        review_rows_123.append({
            "column": column,
            **proposal
        })

review_df = pd.DataFrame(review_rows_123).reset_index(drop=True)

review_index = 0
review_records = []
review_start_time = time.time()

print("Total proposals:", len(review_df))
print(review_df.head())

show_proposal()

In [ ]:
print("review_df columns:", review_df.columns.tolist())
print("review_index:", review_index)
print("review_records:", len(review_records))
print("Current row:")
print(review_df.iloc[review_index].to_dict())

In [ ]:
review_rows_123 = []

for column, proposals in all_llm_proposals_123.items():
    for proposal in proposals:
        review_rows_123.append({
            "column": column,
            **proposal
        })

review_df = pd.DataFrame(review_rows_123).reset_index(drop=True)

print(review_df.columns.tolist())
print("Total:", len(review_df))

In [ ]:
from pathlib import Path
import pandas as pd
import time

project_root = Path.cwd().parent
operational_dir = project_root / "results" / "operational"

proposal_file = operational_dir / "llm_all_proposals_seed_123.csv"

print("Project:", project_root)
print("Proposals file exists:", proposal_file.exists())

In [ ]:
review_df = pd.read_csv(
    operational_dir / "llm_all_proposals_seed_123.csv"
)

column_counts = {
    "job": 70,
    "marital": 20,
    "education": 28,
    "default": 2,
    "housing": 2,
    "loan": 2,
    "contact": 21,
    "month": 10,
    "poutcome": 18
}

review_df.insert(
    0,
    "column",
    [col for col, count in column_counts.items() for _ in range(count)]
)

review_df = review_df.drop(columns=[": "], errors="ignore")

print("Total proposals:", len(review_df))
print(review_df.head())

In [ ]:
import pandas as pd

proposals_df = pd.read_csv(
    operational_dir / "llm_all_proposals_seed_123.csv"
)

review_rows_123 = []

for column, proposals in all_llm_proposals_123.items():
    for proposal in proposals:
        review_rows_123.append({
            "column": column,
            **proposal
        })

review_df = pd.DataFrame(review_rows_123)
review_df = review_df.drop(columns=[": "], errors="ignore")
review_df = review_df.reset_index(drop=True)

print("Total proposals:", len(review_df))
print(review_df.head())

In [ ]:
review_df = pd.read_csv(
    operational_dir / "llm_all_proposals_seed_123.csv"
)

column_counts = {
    "job": 70,
    "marital": 20,
    "education": 28,
    "default": 2,
    "housing": 2,
    "loan": 2,
    "contact": 21,
    "month": 10,
    "poutcome": 18
}

columns = [
    col
    for col, count in column_counts.items()
    for _ in range(count)
]

review_df.insert(0, "column", columns)
review_df = review_df.drop(columns=[": "], errors="ignore")

print(review_df.shape)
print(review_df.head())

In [ ]:
review_records = []
review_start_time = time.time()

for i, row in review_df.iterrows():
    print(f"\nProposal {i+1}/ {len(review_df)}")
    print("Column:", row["column"])
    print("Original:", row["original_value"])
    print("Suggested:", row["suggested_value"])
    print("Reason:", row["reason"])
    print("Confidence:", row["confidence"])

    while True:
        decision = input("Approve (a) / Reject (r): ").strip().lower()
        if decision in ["a", "r"]:
            break

    review_records.append({
        "column": row["column"],
        "original_value": row["original_value"],
        "suggested_value": row["suggested_value"],
        "reason": row["reason"],
        "confidence": row["confidence"],
        "decision": "approve" if decision == "a" else "reject",
        "decision_time": time.strftime("%Y-%m-%d %H:%M:%S")
    })

print("\nReview completed!")
print("Total reviewed:", len(review_records))
print("Approved:", sum(r["decision"] == "approve" for r in review_records))
print("Rejected:", sum(r["decision"] == "reject" for r in review_records))
print("Review time:", round(time.time() - review_start_time, 2), "seconds")

In [ ]:
review_results_123 = pd.DataFrame(review_records)

review_results_123.to_csv(
    operational_dir / "hitl_review_decisions_seed_123.csv",
    index=False
)

hitl_review_metrics_123 = {
    "seed": 123,
    "total_proposals": len(review_records),
    "approved": sum(r["decision"] == "approve" for r in review_records),
    "rejected": sum(r["decision"] == "reject" for r in review_records),
    "review_time_seconds": 237.4
}

pd.DataFrame([hitl_review_metrics_123]).to_csv(
    operational_dir / "hitl_review_metrics_seed_123.csv",
    index=False
)

print("HITL Seed 123 decisions saved successfully.")

In [ ]:
df_corrupted_123 = pd.read_csv(
    project_root / "data" / "corrupted" / "seed_123" / "corrupted_train.csv",
    dtype=str,
    keep_default_na=False
)


all_llm_proposals_123 = {
    col: group.drop(columns=["column"]).to_dict(orient="records")
    for col, group in review_df.groupby("column", sort=False)
}

print("Data loaded:", df_corrupted_123.shape)
print("Proposal columns:", len(all_llm_proposals_123))
print("Total proposals:", sum(map(len, all_llm_proposals_123.values())))

In [ ]:
for path in project_root.rglob("*.py"):
    content = path.read_text(encoding="utf-8")
    if "def execute_categorical_proposals" in content:
        print("Found:", path)

In [ ]:
from src.agents.safe_executor import execute_categorical_proposals

print("Function imported successfully.")

In [ ]:
import sys

sys.path.insert(0, str(project_root))

from src.agents.safe_executor import execute_categorical_proposals

print("Function imported successfully.")

In [ ]:
# Prepare approved proposals
approved_df_123 = review_results_123[
    review_results_123["decision"] == "approve"
]

approved_proposals_123 = {}

for column in approved_df_123["column"].unique():
    approved_proposals_123[column] = (
        approved_df_123[approved_df_123["column"] == column]
        [["original_value", "suggested_value", "reason", "confidence"]]
        .to_dict(orient="records")
    )

# Execute approved proposals only
hitl_cleaned_123 = df_corrupted_123.copy()
hitl_log_123 = []

for column, proposals in approved_proposals_123.items():
    hitl_cleaned_123, log = execute_categorical_proposals(
        dataframe=hitl_cleaned_123,
        column=column,
        valid_proposals=all_llm_proposals_123[column],
        approved_proposals=proposals
    )
    hitl_log_123.extend(log)

# Save cleaned data and execution log
hitl_cleaned_123.to_csv(
    project_root / "data" / "clean" / "hitl_cleaned_seed_123.csv",
    index=False
)

pd.DataFrame(hitl_log_123).to_csv(
    operational_dir / "hitl_execution_log_seed_123.csv",
    index=False
)

print("Approved proposals:", len(approved_df_123))
print("Executed proposals:", len(hitl_log_123))
print("Affected cells:", sum(x["affected_rows"] for x in hitl_log_123))
print("HITL Seed 123 execution saved successfully.")

In [ ]:
# Load clean reference
train_reference = pd.read_csv(
    project_root / "data" / "clean" / "train_clean_reference.csv",
    dtype=str,
    keep_default_na=False
)

original = df_corrupted_123.copy()
cleaned = hitl_cleaned_123.copy()
reference = train_reference.copy()

# Normalize numeric columns
numeric_cols = ["age", "balance", "day", "campaign", "pdays", "previous"]

for col in numeric_cols:
    original[col] = pd.to_numeric(original[col], errors="coerce")
    cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")
    reference[col] = pd.to_numeric(reference[col], errors="coerce")

# Ground-truth corrupted cells
corrupted_keys = set(
    zip(ground_truth_123["row_index"], ground_truth_123["column"])
)

# Detect changed cells safely
changed_keys = set()

for col in original.columns:
    changed_mask = (
        original[col].ne(cleaned[col]).fillna(False)
        & ~(original[col].isna() & cleaned[col].isna())
    )
    changed_keys.update(
        (idx, col) for idx in original.index[changed_mask]
    )

correct_repairs = 0
incorrect_repairs = 0
harmful_changes = 0

for row_idx, col in changed_keys:
    if (row_idx, col) in corrupted_keys:
        if cleaned.at[row_idx, col] == reference.at[row_idx, col]:
            correct_repairs += 1
        else:
            incorrect_repairs += 1
    else:
        harmful_changes += 1

total_corrupted = len(corrupted_keys)
total_changed = len(changed_keys)
missed_corruptions = total_corrupted - len(corrupted_keys & changed_keys)

precision = correct_repairs / total_changed if total_changed else 0
recall = correct_repairs / total_corrupted if total_corrupted else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

hitl_metrics_123 = {
    "seed": 123,
    "total_corrupted": total_corrupted,
    "total_changed": total_changed,
    "correct_repairs": correct_repairs,
    "incorrect_repairs": incorrect_repairs,
    "harmful_clean_changes": harmful_changes,
    "missed_corruptions": missed_corruptions,
    "precision": precision,
    "recall": recall,
    "f1": f1
}

display(pd.DataFrame([hitl_metrics_123]))

pd.DataFrame([hitl_metrics_123]).to_csv(
    project_root / "results" / "operational" / "hitl_metrics_seed_123.csv",
    index=False
)

In [ ]:
ground_truth_123 = pd.read_csv(
    project_root / "data" / "corrupted" / "seed_123" / "ground_truth_mask.csv"
)

print("Ground Truth loaded:", ground_truth_123.shape)

In [ ]:
proposals_path_2026 = operational_dir / "llm_all_proposals_seed_2026.csv"

proposals_2026 = pd.read_csv(proposals_path_2026)

print("Shape:", proposals_2026.shape)
print("\nColumns:")
print(proposals_2026.columns.tolist())

display(proposals_2026.head(10))

In [ ]:
print("Shape:", proposals_2026.shape)
print("Columns:", proposals_2026.columns.tolist())
print("\nMissing values:")
print(proposals_2026.isna().sum())

In [ ]:
train_reference = pd.read_csv(
    project_root / "data" / "clean" / "train_clean_reference.csv",
    dtype=str,
    keep_default_na=False
)

categorical_cols = [
    "job", "marital", "education", "default", "housing",
    "loan", "contact", "month", "poutcome"
]

ALLOWED_VALUES = {
    col: set(train_reference[col].str.strip().str.lower().unique())
    for col in categorical_cols
}

print("ALLOWED_VALUES reconstructed successfully.")

for col, values in ALLOWED_VALUES.items():
    print(f"{col}: {len(values)} allowed values")

In [ ]:
# Map each proposal to its actual column
column_mapping = {}
ambiguous = []

for idx, row in proposals_2026.iterrows():
    value = str(row["original_value"]).strip()

    candidates = [
        col for col in ALLOWED_VALUES
        if value.lower() not in ALLOWED_VALUES[col]
        and (df_corrupted_2026[col].str.strip() == value).any()
    ]

    if len(candidates) == 1:
        column_mapping[idx] = candidates[0]
    else:
        ambiguous.append({
            "index": idx,
            "original_value": value,
            "candidates": candidates
        })

print("Mapped proposals:", len(column_mapping))
print("Ambiguous proposals:", len(ambiguous))

if ambiguous:
    display(pd.DataFrame(ambiguous))

In [ ]:
operational_metrics_2026 = pd.read_csv(
    operational_dir / "llm_operational_metrics_seed_2026.csv"
)

print("Columns:", operational_metrics_2026.columns.tolist())
display(operational_metrics_2026)

In [ ]:
# Reconstruct the original column mapping
column_counts_2026 = {
    "job": 65,
    "marital": 20,
    "education": 28,
    "default": 2,
    "housing": 2,
    "loan": 2,
    "contact": 17,
    "month": 9,
    "poutcome": 18
}

columns_2026 = [
    col
    for col, count in column_counts_2026.items()
    for _ in range(count)
]

review_df_2026 = proposals_2026.drop(
    columns=[": "], errors="ignore"
).copy()

review_df_2026.insert(0, "column", columns_2026)
review_df_2026 = review_df_2026.reset_index(drop=True)

# Save decisions after every response
review_path_2026 = operational_dir / "hitl_review_decisions_seed_2026.csv"

if review_path_2026.exists():
    review_records_2026 = pd.read_csv(review_path_2026).to_dict("records")
else:
    review_records_2026 = []

reviewed_indices = {
    int(r["proposal_index"]) for r in review_records_2026
}

start_time_2026 = time.time()

for idx, row in review_df_2026.iterrows():

    if idx in reviewed_indices:
        continue

    print("\n" + "=" * 60)
    print(f"Proposal {idx + 1}/163")
    print("Column:", row["column"])
    print("Original:", row["original_value"])
    print("Suggested:", row["suggested_value"])
    print("Reason:", row["reason"])
    print("Confidence:", row["confidence"])
    print("-" * 60)

    while True:
        decision = input("Approve (a) / Reject (r) / Quit (q): ").strip().lower()
        if decision in ["a", "r", "q"]:
            break

    if decision == "q":
        break

    review_records_2026.append({
        "proposal_index": idx,
        "column": row["column"],
        "original_value": row["original_value"],
        "suggested_value": row["suggested_value"],
        "reason": row["reason"],
        "confidence": row["confidence"],
        "decision": "approve" if decision == "a" else "reject"
    })

    pd.DataFrame(review_records_2026).to_csv(
        review_path_2026, index=False
    )

print("\nReview progress:", len(review_records_2026), "/ 163")
print("Approved:", sum(r["decision"] == "approve" for r in review_records_2026))
print("Rejected:", sum(r["decision"] == "reject" for r in review_records_2026))
print("Decisions saved.")

In [ ]:
review_results_2026 = pd.DataFrame(review_records_2026)

approved_count = (review_results_2026["decision"] == "approve").sum()
rejected_count = (review_results_2026["decision"] == "reject").sum()
review_time = time.time() - start_time_2026

hitl_review_metrics_2026 = {
    "seed": 2026,
    "total_proposals": len(review_results_2026),
    "approved": approved_count,
    "rejected": rejected_count,
    "review_time_seconds": round(review_time, 2)
}

review_results_2026.to_csv(
    operational_dir / "hitl_review_decisions_seed_2026.csv",
    index=False
)

pd.DataFrame([hitl_review_metrics_2026]).to_csv(
    operational_dir / "hitl_review_metrics_seed_2026.csv",
    index=False
)

display(pd.DataFrame([hitl_review_metrics_2026]))

In [ ]:
# Reconstruct all proposals by column
all_llm_proposals_2026 = {
    col: group.drop(columns=["column"]).to_dict(orient="records")
    for col, group in review_df_2026.groupby("column", sort=False)
}

# Prepare approved proposals
approved_df_2026 = review_results_2026[
    review_results_2026["decision"] == "approve"
]

approved_proposals_2026 = {
    col: group.drop(columns=["column", "decision", "proposal_index"])
             [["original_value", "suggested_value", "reason", "confidence"]]
             .to_dict(orient="records")
    for col, group in approved_df_2026.groupby("column", sort=False)
}

# Execute approved proposals
hitl_cleaned_2026 = df_corrupted_2026.copy()
hitl_log_2026 = []

for column, proposals in approved_proposals_2026.items():
    hitl_cleaned_2026, log = execute_categorical_proposals(
        dataframe=hitl_cleaned_2026,
        column=column,
        valid_proposals=all_llm_proposals_2026[column],
        approved_proposals=proposals
    )
    hitl_log_2026.extend(log)

# Save results
hitl_cleaned_2026.to_csv(
    project_root / "data" / "clean" / "hitl_cleaned_seed_2026.csv",
    index=False
)

pd.DataFrame(hitl_log_2026).to_csv(
    operational_dir / "hitl_execution_log_seed_2026.csv",
    index=False
)

print("Approved proposals:", len(approved_df_2026))
print("Executed proposals:", len(hitl_log_2026))
print("Affected cells:", sum(x["affected_rows"] for x in hitl_log_2026))
print("HITL Seed 2026 execution saved successfully.")

In [ ]:
import pandas as pd
import numpy as np

# Load reference and ground truth
train_reference = pd.read_csv(
    project_root / "data" / "clean" / "train_clean_reference.csv",
    dtype=str,
    keep_default_na=False
)

ground_truth_2026 = pd.read_csv(
    project_root / "data" / "corrupted" / "seed_2026" / "ground_truth_mask.csv"
)

original = df_corrupted_2026.copy()
cleaned = hitl_cleaned_2026.copy()

numeric_cols = ["age", "balance", "day", "campaign", "pdays", "previous"]

# Normalize numeric columns for fair comparison
for col in numeric_cols:
    original[col] = pd.to_numeric(original[col], errors="coerce")
    cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")
    train_reference[col] = pd.to_numeric(train_reference[col], errors="coerce")

# Identify corrupted cells
corrupted_keys = set(
    zip(
        ground_truth_2026["row_index"].astype(int),
        ground_truth_2026["column"]
    )
)

# Identify all changed cells
changed_keys = set()
correct = 0
incorrect = 0
harmful = 0

for col in original.columns:
    changed_mask = (
        original[col].ne(cleaned[col]).fillna(False)
        & ~(original[col].isna() & cleaned[col].isna())
    )

    for row_idx in np.where(changed_mask)[0]:
        key = (int(row_idx), col)
        changed_keys.add(key)

        if key in corrupted_keys:
            if cleaned.at[row_idx, col] == train_reference.at[row_idx, col]:
                correct += 1
            else:
                incorrect += 1
        else:
            harmful += 1

total_corrupted = len(corrupted_keys)
total_changed = len(changed_keys)
missed = total_corrupted - len(corrupted_keys & changed_keys)

precision = correct / total_changed if total_changed else 0
recall = correct / total_corrupted if total_corrupted else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

hitl_metrics_2026 = {
    "seed": 2026,
    "total_corrupted": total_corrupted,
    "total_changed": total_changed,
    "correct": correct,
    "incorrect": incorrect,
    "harmful": harmful,
    "missed": missed,
    "precision": precision,
    "recall": recall,
    "f1": f1
}

pd.DataFrame([hitl_metrics_2026]).to_csv(
    operational_dir / "hitl_metrics_seed_2026.csv",
    index=False
)

display(pd.DataFrame([hitl_metrics_2026]))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent

data_files = [
    "data/clean/train_clean_reference.csv",
    "data/clean/test_clean.csv",
]

for seed in [42, 123, 2026]:
    data_files.extend([
        f"data/corrupted/seed_{seed}/corrupted_train.csv",
        f"data/clean/autonomous_cleaned_seed_{seed}.csv",
        f"data/clean/hitl_cleaned_seed_{seed}.csv",
    ])

for file in data_files:
    path = project_root / file
    print(("FOUND " if path.exists() else "MISSING ") + file)

print("\nClean reference:", 
      pd.read_csv(project_root / "data/clean/train_clean_reference.csv").shape)

print("Clean test:",
      pd.read_csv(project_root / "data/clean/test_clean.csv").shape)

In [ ]:
for seed in [42, 123, 2026]:
    path = project_root / "data" / "clean" / f"rule_based_cleaned_seed_{seed}.csv"
    print(("FOUND " if path.exists() else "MISSING ") + str(path.name))

In [ ]:
for folder in [
    project_root / "data" / "clean",
    project_root / "results" / "cleaning"
]:
    print(f"\n--- {folder} ---")
    if folder.exists():
        for file in sorted(folder.glob("*.csv")):
            print(file.name)

In [ ]:
cleaning_dir = project_root / "results" / "cleaning"

for seed in [42, 123, 2026]:
    path = cleaning_dir / f"rule_based_cleaned_seed_{seed}.csv"

    if path.exists():
        df = pd.read_csv(path, dtype=str, keep_default_na=False)
        print(f"Seed {seed}: FOUND | Shape: {df.shape}")
    else:
        print(f"Seed {seed}: NOT FOUND")

print("\nSearching for Seed 42 Rule-Based file:")
for path in project_root.rglob("*42*.csv"):
    if "rule" in path.name.lower():
        print(path.relative_to(project_root))

In [ ]:
print("rule_based_clean available:", "rule_based_clean" in globals())
print("NUMERICAL_COLS available:", "NUMERICAL_COLS" in globals())
print("VALID_CATEGORIES available:", "VALID_CATEGORIES" in globals())
print("DOMAIN_RULES available:", "DOMAIN_RULES" in globals())

In [ ]:
for folder_name in ["cleaning", "evaluation", "corruption"]:
    folder = project_root / "src" / folder_name

    print(f"\n--- src/{folder_name} ---")

    if folder.exists():
        for file in sorted(folder.glob("*.py")):
            print(file.name)

In [ ]:
cleaner_path = project_root / "src" / "cleaning" / "rule_based_cleaner.py"

print(cleaner_path.read_text(encoding="utf-8"))

In [ ]:
import sys

sys.path.insert(0, str(project_root))

from src.cleaning.rule_based_cleaner import rule_based_clean

print("Rule-Based cleaner imported successfully!")

In [ ]:
print("Reading file...")

cleaner_path = project_root / "src" / "cleaning" / "rule_based_cleaner.py"

with open(cleaner_path, "r", encoding="utf-8") as f:
    content = f.read()

print("File length:", len(content))
print(content[:10000])

In [ ]:
notebooks_dir = project_root / "notebooks"

for file in notebooks_dir.glob("*.ipynb"):
    print(file.name)

In [ ]:
import json

notebook_path = project_root / "notebooks" / "03_rule_based_cleaning.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])
        if any(word in source.lower() for word in [
            "def ", "rule_based", "domain_rules", "valid_categories"
        ]):
            print(f"\n{'='*60}\nCELL {i}\n{'='*60}")
            print(source[:5000])

In [ ]:
import pandas as pd
import numpy as np
from difflib import get_close_matches

# Original categorical definitions
VALID_CATEGORIES = {
    "job": [
        "admin.", "blue-collar", "entrepreneur", "housemaid",
        "management", "retired", "self-employed", "services",
        "student", "technician", "unemployed", "unknown"
    ],
    "marital": ["divorced", "married", "single"],
    "education": ["primary", "secondary", "tertiary", "unknown"],
    "default": ["yes", "no"],
    "housing": ["yes", "no"],
    "loan": ["yes", "no"],
    "contact": ["cellular", "telephone", "unknown"],
    "month": [
        "jan", "feb", "mar", "apr", "may", "jun",
        "jul", "aug", "sep", "oct", "nov", "dec"
    ],
    "poutcome": ["failure", "other", "success", "unknown"]
}

NUMERICAL_COLS = [
    "age", "balance", "day", "campaign",
    "pdays", "previous"
]

DOMAIN_RULES = {
    "age": lambda x: 18 <= x <= 100,
    "day": lambda x: 1 <= x <= 31,
    "campaign": lambda x: x >= 1,
    "pdays": lambda x: x >= -1,
    "previous": lambda x: x >= 0
}

# Original Rule-Based cleaning function
def rule_based_clean(data):
    cleaned = data.copy()
    repair_records = []
    original_data = data.copy()

    for col, valid_values in VALID_CATEGORIES.items():
        for idx in cleaned.index:
            value = cleaned.at[idx, col]

            if pd.isna(value):
                continue

            original_value = value
            normalized = str(value).strip().lower()
            repaired = normalized
            operation = None

            if normalized != str(value):
                operation = "format_normalization"

            if repaired not in valid_values:
                matches = get_close_matches(
                    repaired, valid_values, n=1, cutoff=0.80
                )
                if matches:
                    repaired = matches[0]
                    operation = "fuzzy_matching"

            if repaired != normalized or operation == "format_normalization":
                cleaned.at[idx, col] = repaired
                repair_records.append({
                    "row_index": idx,
                    "column": col,
                    "original_value": original_value,
                    "repaired_value": repaired,
                    "operation": operation
                })

        mode_values = cleaned[col].dropna().mode()

        if not mode_values.empty:
            mode_value = mode_values.iloc[0]
            missing_indices = cleaned.index[cleaned[col].isna()]

            for idx in missing_indices:
                cleaned.at[idx, col] = mode_value
                repair_records.append({
                    "row_index": idx,
                    "column": col,
                    "original_value": np.nan,
                    "repaired_value": mode_value,
                    "operation": "categorical_mode_imputation"
                })

    for col in NUMERICAL_COLS:
        cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")

        if col in DOMAIN_RULES:
            invalid_mask = (
                cleaned[col].notna()
                & ~cleaned[col].apply(DOMAIN_RULES[col])
            )
        else:
            invalid_mask = pd.Series(False, index=cleaned.index)

        cleaned.loc[invalid_mask, col] = np.nan
        median_value = cleaned[col].median()

        if pd.notna(median_value):
            missing_indices = cleaned.index[cleaned[col].isna()]

            for idx in missing_indices:
                original_value = original_data.at[idx, col]

                operation = (
                    "numerical_median_imputation"
                    if pd.isna(original_value)
                    else "invalid_value_median_imputation"
                )

                cleaned.at[idx, col] = median_value
                repair_records.append({
                    "row_index": idx,
                    "column": col,
                    "original_value": original_value,
                    "repaired_value": median_value,
                    "operation": operation
                })

    return cleaned, pd.DataFrame(repair_records)


# Load Seed 42
df_corrupted_42 = pd.read_csv(
    project_root / "data" / "corrupted" / "seed_42" / "corrupted_train.csv",
    dtype=str,
    keep_default_na=False,
    na_values=[""]
)

# Apply original cleaner
df_rule_42, repair_log_42 = rule_based_clean(df_corrupted_42.copy())

# Save cleaned data
output_path = project_root / "data" / "clean" / "rule_based_cleaned_seed_42.csv"

df_rule_42.to_csv(output_path, index=False)

print("Seed 42 Rule-Based cleaning completed.")
print("Cleaned shape:", df_rule_42.shape)
print("Repair operations:", len(repair_log_42))
print("Saved:", output_path)
print("\nOperation distribution:")
display(repair_log_42["operation"].value_counts())

In [23]:
# Load clean test and reference training data
clean_train = pd.read_csv(
    project_root / "data/clean/train_clean_reference.csv",
    dtype=str, keep_default_na=False, na_values=[""]
)

clean_test = pd.read_csv(
    project_root / "data/clean/test_clean.csv",
    dtype=str, keep_default_na=False, na_values=[""]
)

# Define numerical and categorical features
NUMERICAL_COLS = [
    "age", "balance", "day", "campaign", "pdays", "previous"
]

CATEGORICAL_COLS = [
    "job", "marital", "education", "default", "housing",
    "loan", "contact", "month", "poutcome"
]

TARGET = "y"

# Load all training versions
training_data = {"clean_reference": clean_train}

for seed in [42, 123, 2026]:

    base = project_root / "data"

    training_data[f"corrupted_{seed}"] = pd.read_csv(
        base / f"corrupted/seed_{seed}/corrupted_train.csv",
        dtype=str, keep_default_na=False, na_values=[""]
    )

    rule_path = (
        base / f"clean/rule_based_cleaned_seed_{seed}.csv"
        if seed == 42
        else project_root / f"results/cleaning/rule_based_cleaned_seed_{seed}.csv"
    )

    training_data[f"rule_based_{seed}"] = pd.read_csv(
        rule_path, dtype=str, keep_default_na=False, na_values=[""]
    )

    for method in ["autonomous", "hitl"]:
        training_data[f"{method}_{seed}"] = pd.read_csv(
            base / f"clean/{method}_cleaned_seed_{seed}.csv",
            dtype=str, keep_default_na=False, na_values=[""]
        )

# Validate schemas and target distributions
print("TEST SHAPE:", clean_test.shape)
print("TRAINING VERSIONS:", len(training_data))

for name, df in training_data.items():
    assert df.shape == (4000, 16), f"Unexpected shape: {name}"
    assert list(df.columns) == list(clean_test.columns), f"Column mismatch: {name}"
    assert TARGET in df.columns

    print(
        f"{name:22} | Shape: {df.shape} | "
        f"Target counts: {df[TARGET].value_counts().to_dict()}"
    )

# Validate feature definitions
features = NUMERICAL_COLS + CATEGORICAL_COLS

assert len(features) == 15
assert "duration" not in features
assert set(features + [TARGET, "duration"]) == set(clean_test.columns)

print("\nSchema validation passed.")
print("Numerical features:", len(NUMERICAL_COLS))
print("Categorical features:", len(CATEGORICAL_COLS))
print("Excluded feature: duration")

NameError: name 'project_root' is not defined

In [24]:
print("Actual columns:")
for i, col in enumerate(clean_test.columns, 1):
    print(i, repr(col))

print("\nExpected features missing from data:")
print(set(NUMERICAL_COLS + CATEGORICAL_COLS) - set(clean_test.columns))

print("\nColumns not included in feature lists:")
print(set(clean_test.columns) - set(NUMERICAL_COLS + CATEGORICAL_COLS + [TARGET]))

Actual columns:


NameError: name 'clean_test' is not defined

In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Prepare features and target
FEATURES = NUMERICAL_COLS + CATEGORICAL_COLS

X_test = clean_test[FEATURES].copy()
y_test = clean_test[TARGET].map({"no": 0, "yes": 1})

# Ensure numeric columns have numeric types
for col in NUMERICAL_COLS:
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, NUMERICAL_COLS),
    ("categorical", categorical_transformer, CATEGORICAL_COLS)
])

# Fixed Random Forest model
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

# Clean reference training
X_train = clean_train[FEATURES].copy()
y_train = clean_train[TARGET].map({"no": 0, "yes": 1})

for col in NUMERICAL_COLS:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")

# Train and evaluate
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

clean_reference_results = {
    "dataset": "clean_reference",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "pr_auc": average_precision_score(y_test, y_prob)
}

display(pd.DataFrame([clean_reference_results]).round(4))

NameError: name 'CATEGORICAL_COLS' is not defined

In [ ]:
classification_results = [clean_reference_results.copy()]

for name, train_df in training_data.items():

    if name == "clean_reference":
        continue

    X_train = train_df[FEATURES].copy()
    y_train = train_df[TARGET].map({"no": 0, "yes": 1})

    for col in NUMERICAL_COLS:
        X_train[col] = pd.to_numeric(X_train[col], errors="coerce")

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    classification_results.append({
        "dataset": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
        "pr_auc": average_precision_score(y_test, probabilities)
    })

    print(f"Completed: {name}")

classification_df = pd.DataFrame(classification_results)

display(
    classification_df
    .sort_values("dataset")
    .round(4)
)

In [ ]:
# Extract method and seed from dataset names
summary_df = classification_df.copy()

summary_df["method"] = summary_df["dataset"].str.replace(
    r"_\d+$", "", regex=True
)

summary_df["seed"] = summary_df["dataset"].str.extract(
    r"_(42|123|2026)$"
)

METRICS = [
    "accuracy", "precision", "recall",
    "f1", "roc_auc", "pr_auc"
]

# Aggregate results across the three seeds
method_summary = (
    summary_df[summary_df["method"] != "clean_reference"]
    .groupby("method")[METRICS]
    .agg(["mean", "std"])
    .round(4)
)

# Save detailed and summary results
classification_df.to_csv(
    project_root / "results/classification/classification_all_results.csv",
    index=False
)

method_summary.to_csv(
    project_root / "results/classification/classification_method_summary.csv"
)

display(method_summary)
print("\nResults saved successfully.")

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_transformer_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_lr = ColumnTransformer([
    ("numeric", numeric_transformer_lr, NUMERICAL_COLS),
    ("categorical", categorical_transformer_lr, CATEGORICAL_COLS)
])

lr_model_template = Pipeline([
    ("preprocessor", preprocessor_lr),
    ("classifier", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42,
        solver="lbfgs"
    ))
])

print("Scaled Logistic Regression pipeline is ready.")

In [ ]:
lr_results = []

for name, train_df in training_data.items():

    X_train = train_df[FEATURES].copy()
    y_train = train_df[TARGET].map({"no": 0, "yes": 1})

    for col in NUMERICAL_COLS:
        X_train[col] = pd.to_numeric(X_train[col], errors="coerce")

    model = lr_model_template
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    lr_results.append({
        "dataset": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
        "pr_auc": average_precision_score(y_test, probabilities)
    })

    print(f"Completed: {name}")

lr_df = pd.DataFrame(lr_results)

lr_df.to_csv(
    project_root / "results/classification/logistic_regression_results.csv",
    index=False
)

display(lr_df.round(4))

In [ ]:
# Extract method and seed
lr_summary_df = lr_df.copy()

lr_summary_df["method"] = lr_summary_df["dataset"].str.replace(
    r"_\d+$", "", regex=True
)

lr_summary_df["seed"] = lr_summary_df["dataset"].str.extract(
    r"_(42|123|2026)$"
)

METRICS = [
    "accuracy", "precision", "recall",
    "f1", "roc_auc", "pr_auc"
]

# Aggregate across seeds
lr_method_summary = (
    lr_summary_df[lr_summary_df["method"] != "clean_reference"]
    .groupby("method")[METRICS]
    .agg(["mean", "std"])
    .round(4)
)

# Save summary
lr_method_summary.to_csv(
    project_root / "results/classification/logistic_regression_method_summary.csv"
)

display(lr_method_summary)
print("\nLogistic Regression summary saved.")

In [ ]:
from sklearn.dummy import DummyClassifier

# Majority-class baseline
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)

majority_results = {
    "model": "Majority Class Baseline",
    "accuracy": accuracy_score(y_test, dummy_pred),
    "precision": precision_score(y_test, dummy_pred, zero_division=0),
    "recall": recall_score(y_test, dummy_pred, zero_division=0),
    "f1": f1_score(y_test, dummy_pred, zero_division=0),
    "roc_auc": float("nan"),
    "pr_auc": float("nan")
}

majority_df = pd.DataFrame([majority_results])

majority_df.to_csv(
    project_root / "results/classification/majority_baseline.csv",
    index=False
)

display(majority_df.round(4))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(4))

In [ ]:
classification_dir = project_root / "results" / "classification"

print("Classification result files:")
for file in sorted(classification_dir.glob("*.csv")):
    print(f"{file.name} | {file.stat().st_size:,} bytes")

print("\nTotal CSV files:", len(list(classification_dir.glob("*.csv"))))

In [ ]:
for folder_name in ["cleaning", "operational"]:
    folder = project_root / "results" / folder_name

    print(f"\n{'='*20} {folder_name.upper()} {'='*20}")

    for file in sorted(folder.glob("*.csv")):
        print(f"{file.name} | {file.stat().st_size:,} bytes")

In [ ]:
import pandas as pd

folders = [
    project_root / "results" / "cleaning",
    project_root / "results" / "operational"
]

for folder in folders:
    print(f"\n{'='*25} {folder.name.upper()} {'='*25}")

    files = sorted(folder.glob("*metrics*.csv"))

    if folder.name == "cleaning":
        files += [
            folder / "rule_based_overall_by_seed_corrected.csv",
            folder / "rule_based_summary_statistics.csv"
        ]

    for file in files:
        if not file.exists():
            continue

        print(f"\n--- {file.name} ---")
        df = pd.read_csv(file)
        print(df.to_string(index=False))

In [ ]:
import pandas as pd

files_to_check = [
    "cleaning/rule_based_metrics_seed_42.csv",
    "cleaning/rule_based_metrics_seed_123.csv",
    "cleaning/rule_based_metrics_seed_2026.csv",
    "cleaning/rule_based_overall_by_seed_corrected.csv",
    "cleaning/rule_based_overall_results.csv",
    "cleaning/hitl_metrics_seed_42.csv",
    "operational/autonomous_metrics_seed_123.csv",
    "operational/autonomous_metrics_seed_2026.csv",
    "operational/hitl_metrics_seed_123.csv",
    "operational/hitl_metrics_seed_2026.csv",
    "operational/hitl_review_seed_42.csv",
]

for relative_path in files_to_check:
    file_path = project_root / "results" / relative_path

    print(f"\n{'='*65}")
    print(relative_path)

    if file_path.exists():
        df = pd.read_csv(file_path)
        print(df.to_string(index=False))
    else:
        print("FILE NOT FOUND")

In [26]:
import pandas as pd
import numpy as np

seeds = [42, 123, 2026]

reference_path = project_root / "data" / "clean" / "train_clean_reference.csv"
reference = pd.read_csv(reference_path, dtype=str, keep_default_na=False)

def find_output(method, seed):
    candidates = {
        "rule_based": [
            project_root / "data" / "clean" / f"rule_based_cleaned_seed_{seed}.csv",
            project_root / "results" / "cleaning" / f"rule_based_cleaned_seed_{seed}.csv",
        ],
        "autonomous": [
            project_root / "data" / "clean" / f"autonomous_cleaned_seed_{seed}.csv"
        ],
        "hitl": [
            project_root / "data" / "clean" / f"hitl_cleaned_seed_{seed}.csv"
        ],
    }

    for path in candidates[method]:
        if path.exists():
            return path

    return None


results = []

for seed in seeds:
    mask_path = (
        project_root / "data" / "corrupted" /
        f"seed_{seed}" / "ground_truth_mask.csv"
    )

    mask_df = pd.read_csv(mask_path, dtype=str, keep_default_na=False)

    mask_lookup = {
        (int(row.row_index), row.column): row.original_value
        for row in mask_df.itertuples(index=False)
    }

    for method in ["rule_based", "autonomous", "hitl"]:
        output_path = find_output(method, seed)

        if output_path is None:
            print(f"MISSING: {method}, seed {seed}")
            continue

        cleaned = pd.read_csv(
            output_path, dtype=str, keep_default_na=False
        )

        if cleaned.shape != reference.shape or list(cleaned.columns) != list(reference.columns):
            print(f"SHAPE/COLUMN MISMATCH: {method}, seed {seed}")
            continue

        changed_positions = np.where(
            reference.ne(cleaned).to_numpy()
        )

        changed_keys = {
            (int(row), reference.columns[col])
            for row, col in zip(*changed_positions)
        }

        correct = 0
        incorrect = 0
        harmful = 0

        for row, col in changed_keys:
            key = (row, col)

            if key in mask_lookup:
                if cleaned.at[row, col] == mask_lookup[key]:
                    correct += 1
                else:
                    incorrect += 1
            else:
                harmful += 1

        missed = len(mask_lookup) - correct - incorrect
        total_changed = len(changed_keys)

        precision = correct / total_changed if total_changed else 0
        recall = correct / len(mask_lookup) if mask_lookup else 0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0
        )

        results.append({
            "seed": seed,
            "method": method,
            "total_corrupted": len(mask_lookup),
            "total_changed": total_changed,
            "correct_repairs": correct,
            "incorrect_repairs": incorrect,
            "harmful_changes": harmful,
            "missed_corruptions": missed,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

recomputed_metrics = pd.DataFrame(results)

print("\nRECOMPUTED REPAIR METRICS")
display(recomputed_metrics.round(4))

NameError: name 'project_root' is not defined

In [ ]:
seed = 42

reference = pd.read_csv(
    project_root / "data/clean/train_clean_reference.csv",
    dtype=str, keep_default_na=False
)

corrupted = pd.read_csv(
    project_root / f"data/corrupted/seed_{seed}/corrupted_train.csv",
    dtype=str, keep_default_na=False
)

mask = pd.read_csv(
    project_root / f"data/corrupted/seed_{seed}/ground_truth_mask.csv",
    dtype=str, keep_default_na=False
)

rule_path = find_output("rule_based", seed)
rule_cleaned = pd.read_csv(
    rule_path, dtype=str, keep_default_na=False
)

print("SHAPES")
print("Reference:", reference.shape)
print("Corrupted:", corrupted.shape)
print("Rule cleaned:", rule_cleaned.shape)
print("Mask:", mask.shape)

print("\nCOLUMNS MATCH")
print("Reference vs corrupted:", list(reference.columns) == list(corrupted.columns))
print("Reference vs cleaned:", list(reference.columns) == list(rule_cleaned.columns))

print("\nFIRST ROWS")
display(reference.head(3))
display(corrupted.head(3))
display(rule_cleaned.head(3))

print("\nCHECK 10 GROUND-TRUTH RECORDS")
for _, item in mask.head(10).iterrows():
    row = int(item["row_index"])
    col = item["column"]

    print({
        "row": row,
        "column": col,
        "mask_original": item["original_value"],
        "reference_value": reference.at[row, col],
        "corrupted_value": corrupted.at[row, col],
        "cleaned_value": rule_cleaned.at[row, col],
    })

In [ ]:
import pandas as pd
import numpy as np

seeds = [42, 123, 2026]

numeric_cols = [
    "age", "balance", "day", "campaign", "pdays", "previous"
]

def normalize_value(value, column):
    if pd.isna(value) or str(value).strip() == "":
        return None

    if column in numeric_cols:
        try:
            return float(value)
        except (ValueError, TypeError):
            return str(value).strip()

    return str(value).strip()


results = []

for seed in seeds:

    corrupted = pd.read_csv(
        project_root / "data" / "corrupted" /
        f"seed_{seed}" / "corrupted_train.csv",
        dtype=str, keep_default_na=False
    )

    mask_df = pd.read_csv(
        project_root / "data" / "corrupted" /
        f"seed_{seed}" / "ground_truth_mask.csv",
        dtype=str, keep_default_na=False
    )

    mask_lookup = {
        (int(row.row_index), row.column):
        normalize_value(row.original_value, row.column)
        for row in mask_df.itertuples(index=False)
    }

    for method in ["rule_based", "autonomous", "hitl"]:

        output_path = find_output(method, seed)

        if output_path is None:
            print(f"MISSING: {method}, seed {seed}")
            continue

        cleaned = pd.read_csv(
            output_path, dtype=str, keep_default_na=False
        )

        correct = 0
        incorrect = 0
        harmful = 0

        for row in range(len(corrupted)):
            for col in corrupted.columns:

                corrupted_value = normalize_value(
                    corrupted.at[row, col], col
                )

                cleaned_value = normalize_value(
                    cleaned.at[row, col], col
                )

                # Did the cleaning method change this cell?
                if corrupted_value == cleaned_value:
                    continue

                key = (row, col)

                if key in mask_lookup:

                    if cleaned_value == mask_lookup[key]:
                        correct += 1
                    else:
                        incorrect += 1

                else:
                    harmful += 1

        total_corrupted = len(mask_lookup)
        total_changed = correct + incorrect + harmful
        missed = total_corrupted - correct - incorrect

        precision = (
            correct / total_changed if total_changed else 0
        )

        recall = (
            correct / total_corrupted if total_corrupted else 0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0
        )

        results.append({
            "seed": seed,
            "method": method,
            "total_corrupted": total_corrupted,
            "total_changed": total_changed,
            "correct_repairs": correct,
            "incorrect_repairs": incorrect,
            "harmful_changes": harmful,
            "missed_corruptions": missed,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

recomputed_metrics = pd.DataFrame(results)

display(recomputed_metrics.round(4))

In [ ]:
seed = 123

candidates = [
    project_root / "data" / "clean" /
    f"rule_based_cleaned_seed_{seed}.csv",

    project_root / "results" / "cleaning" /
    f"rule_based_cleaned_seed_{seed}.csv",
]

for path in candidates:
    print("\nFILE:", path)
    print("EXISTS:", path.exists())

    if path.exists():
        df = pd.read_csv(
            path, dtype=str, keep_default_na=False
        )
        print("SHAPE:", df.shape)

        # Compare this output against corrupted data
        corrupted = pd.read_csv(
            project_root / "data" / "corrupted" /
            f"seed_{seed}" / "corrupted_train.csv",
            dtype=str, keep_default_na=False
        )

        changed = 0
        for col in corrupted.columns:
            changed += (
                corrupted[col].str.strip()
                != df[col].str.strip()
            ).sum()

        print("RAW STRING CHANGES:", changed)

In [ ]:
seed = 123

reference = pd.read_csv(
    project_root / "data/clean/train_clean_reference.csv",
    dtype=str, keep_default_na=False
)

corrupted = pd.read_csv(
    project_root / "data/corrupted/seed_123/corrupted_train.csv",
    dtype=str, keep_default_na=False
)

cleaned = pd.read_csv(
    project_root / "results/cleaning/rule_based_cleaned_seed_123.csv",
    dtype=str, keep_default_na=False
)

mask_df = pd.read_csv(
    project_root / "data/corrupted/seed_123/ground_truth_mask.csv",
    dtype=str, keep_default_na=False
)

mask_keys = {
    (int(r.row_index), r.column)
    for r in mask_df.itertuples(index=False)
}

def norm(v, col):
    if pd.isna(v) or str(v).strip() == "":
        return None
    if col in numeric_cols:
        try:
            return float(v)
        except:
            return str(v).strip()
    return str(v).strip()

harmful_examples = []

for row in range(len(corrupted)):
    for col in corrupted.columns:
        before = norm(corrupted.at[row, col], col)
        after = norm(cleaned.at[row, col], col)

        if before != after and (row, col) not in mask_keys:
            harmful_examples.append({
                "row": row,
                "column": col,
                "reference": reference.at[row, col],
                "corrupted": corrupted.at[row, col],
                "cleaned": cleaned.at[row, col]
            })

harmful_df = pd.DataFrame(harmful_examples)

print("HARMFUL CHANGES:", len(harmful_df))
print("\nHARMFUL CHANGES BY COLUMN:")
display(harmful_df["column"].value_counts().to_frame("count"))

print("\nFIRST 25 HARMFUL EXAMPLES:")
display(harmful_df.head(25))

In [ ]:
for folder_name in ["results/cleaning", "logs"]:
    folder = project_root / folder_name

    print(f"\n--- {folder_name} ---")

    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if path.is_file() and (
                "rule" in path.name.lower()
                or "clean" in path.name.lower()
                or "repair" in path.name.lower()
            ):
                print(path.relative_to(project_root), path.stat().st_size)

In [ ]:
import pandas as pd
import numpy as np

def norm(v, col):
    if pd.isna(v) or str(v).strip() == "":
        return None
    if col in numeric_cols:
        try:
            return float(v)
        except (ValueError, TypeError):
            return str(v).strip()
    return str(v).strip()


candidates = {
    42: [
        project_root / "results/cleaning/seed_42/rule_based_cleaned.csv",
        project_root / "results/cleaning/seed_42/rule_based_cleaned_corrected.csv",
        project_root / "data/clean/rule_based_cleaned_seed_42.csv",
    ],
    123: [
        project_root / "results/cleaning/rule_based_cleaned_seed_123.csv",
        project_root / "results/cleaning/seed_123/rule_based_cleaned.csv",
        project_root / "results/cleaning/seed_123/rule_based_cleaned_corrected.csv",
    ],
    2026: [
        project_root / "results/cleaning/rule_based_cleaned_seed_2026.csv",
        project_root / "results/cleaning/seed_2026/rule_based_cleaned.csv",
        project_root / "results/cleaning/seed_2026/rule_based_cleaned_corrected.csv",
    ],
}

comparison_results = []

for seed, paths in candidates.items():

    corrupted = pd.read_csv(
        project_root / "data/corrupted" /
        f"seed_{seed}/corrupted_train.csv",
        dtype=str, keep_default_na=False
    )

    mask_df = pd.read_csv(
        project_root / "data/corrupted" /
        f"seed_{seed}/ground_truth_mask.csv",
        dtype=str, keep_default_na=False
    )

    mask_lookup = {
        (int(r.row_index), r.column):
        norm(r.original_value, r.column)
        for r in mask_df.itertuples(index=False)
    }

    for path in paths:

        if not path.exists():
            continue

        cleaned = pd.read_csv(
            path, dtype=str, keep_default_na=False
        )

        correct = incorrect = harmful = 0

        for row in range(len(corrupted)):
            for col in corrupted.columns:

                before = norm(corrupted.at[row, col], col)
                after = norm(cleaned.at[row, col], col)

                if before == after:
                    continue

                key = (row, col)

                if key in mask_lookup:
                    if after == mask_lookup[key]:
                        correct += 1
                    else:
                        incorrect += 1
                else:
                    harmful += 1

        changed = correct + incorrect + harmful

        comparison_results.append({
            "seed": seed,
            "file": str(path.relative_to(project_root)),
            "correct": correct,
            "incorrect": incorrect,
            "harmful": harmful,
            "total_changed": changed,
        })

display(pd.DataFrame(comparison_results))


# Inspect repair logs
for seed in [42, 123, 2026]:

    log_path = (
        project_root / "results/cleaning" /
        f"seed_{seed}/rule_based_repair_log.csv"
    )

    print(f"\n--- SEED {seed} REPAIR LOG ---")

    if log_path.exists():
        log = pd.read_csv(log_path)

        print("Columns:", log.columns.tolist())
        print("Rows:", len(log))
        display(log.head(8))
    else:
        print("Repair log not found.")
        

In [ ]:
# Explicitly select the verified Rule-Based outputs
rule_based_paths = {
    42: project_root / "results/cleaning/seed_42/rule_based_cleaned.csv",
    123: project_root / "results/cleaning/seed_123/rule_based_cleaned.csv",
    2026: project_root / "results/cleaning/seed_2026/rule_based_cleaned.csv",
}

final_results = []

for seed in [42, 123, 2026]:

    corrupted = pd.read_csv(
        project_root / "data/corrupted" /
        f"seed_{seed}/corrupted_train.csv",
        dtype=str, keep_default_na=False
    )

    mask_df = pd.read_csv(
        project_root / "data/corrupted" /
        f"seed_{seed}/ground_truth_mask.csv",
        dtype=str, keep_default_na=False
    )

    mask_lookup = {
        (int(r.row_index), r.column):
        norm(r.original_value, r.column)
        for r in mask_df.itertuples(index=False)
    }

    for method in ["rule_based", "autonomous", "hitl"]:

        if method == "rule_based":
            output_path = rule_based_paths[seed]
        else:
            output_path = find_output(method, seed)

        cleaned = pd.read_csv(
            output_path, dtype=str, keep_default_na=False
        )

        correct = incorrect = harmful = 0

        for row in range(len(corrupted)):
            for col in corrupted.columns:

                before = norm(corrupted.at[row, col], col)
                after = norm(cleaned.at[row, col], col)

                if before == after:
                    continue

                if (row, col) in mask_lookup:
                    if after == mask_lookup[(row, col)]:
                        correct += 1
                    else:
                        incorrect += 1
                else:
                    harmful += 1

        total_changed = correct + incorrect + harmful
        total_corrupted = len(mask_lookup)
        missed = total_corrupted - correct - incorrect

        precision = correct / total_changed if total_changed else 0
        recall = correct / total_corrupted if total_corrupted else 0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0
        )

        final_results.append({
            "seed": seed,
            "method": method,
            "total_corrupted": total_corrupted,
            "total_changed": total_changed,
            "correct_repairs": correct,
            "incorrect_repairs": incorrect,
            "harmful_changes": harmful,
            "missed_corruptions": missed,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

final_metrics = pd.DataFrame(final_results)

summary = (
    final_metrics
    .groupby("method")[["precision", "recall", "f1"]]
    .agg(["mean", "std"])
)

final_metrics.to_csv(
    project_root / "results/cleaning/final_repair_metrics.csv",
    index=False
)

summary.to_csv(
    project_root / "results/cleaning/final_repair_summary.csv"
)

print("FINAL REPAIR METRICS")
display(final_metrics.round(4))

print("\nMEAN ± SD")
display(summary.round(4))

print("\nSaved final evaluation files.")

In [ ]:
# Search for all HITL Seed 42 files
search_folders = [
    project_root / "results",
    project_root / "data",
    project_root / "logs",
]

for folder in search_folders:
    print(f"\n--- {folder.name} ---")

    for path in folder.rglob("*"):
        if path.is_file() and "hitl" in path.name.lower() and "42" in path.name:
            print(f"\nFile: {path.relative_to(project_root)}")
            print(f"Size: {path.stat().st_size} bytes")

            if path.suffix.lower() == ".csv" and path.stat().st_size > 2:
                try:
                    df = pd.read_csv(path)
                    print("Columns:", df.columns.tolist())
                    print("Rows:", len(df))
                    display(df.head(5))
                except Exception as e:
                    print("Could not read:", e)

In [ ]:
import json

execution_path = (
    project_root / "results/operational/"
    "hitl_execution_log_seed_42.json"
)

with open(execution_path, "r", encoding="utf-8") as f:
    execution_data = json.load(f)

print("DATA TYPE:", type(execution_data))

if isinstance(execution_data, dict):
    print("\nTOP-LEVEL KEYS:")
    print(list(execution_data.keys()))

    for key, value in execution_data.items():
        print(f"\n--- {key} ---")
        if isinstance(value, list):
            print("Items:", len(value))
            display(pd.DataFrame(value).head(5))
        elif isinstance(value, dict):
            print(value)
        else:
            print(value)

elif isinstance(execution_data, list):
    print("Items:", len(execution_data))
    display(pd.DataFrame(execution_data).head(10))

In [ ]:
execution_df = pd.DataFrame(execution_data)

print("EXECUTION LOG SUMMARY")
print("Total proposals:", len(execution_df))
print("\nStatus distribution:")
display(execution_df["status"].value_counts().to_frame("count"))

print("\nTotal affected cells:")
print(execution_df["affected_cells"].sum())

print("\nUnique columns:")
print(execution_df["column"].nunique())

print("\nRows by status:")
display(
    execution_df.groupby("status").agg(
        proposals=("column", "count"),
        affected_cells=("affected_cells", "sum")
    )
)

print("\nCheck executed operations:")
display(
    execution_df[
        ["column", "original_value", "suggested_value",
         "affected_cells", "status"]
    ].tail(10)
)

In [ ]:
repair_results_path = (
    project_root / "results" / "cleaning"
)

repair_tables = []

for seed in [42, 123, 2026]:
    for method in ["rule_based", "autonomous", "hitl"]:

        if method == "rule_based":
            file_path = (
                repair_results_path
                / f"repair_metrics_seed_{seed}.csv"
            )
        else:
            file_path = (
                repair_results_path
                / f"{method}_metrics_seed_{seed}.csv"
            )

        if file_path.exists():
            df = pd.read_csv(file_path)
            df["method"] = method
            repair_tables.append(df)
            print("Loaded:", file_path.name)
        else:
            print("Not found:", file_path.name)

if repair_tables:
    final_repair_results = pd.concat(
        repair_tables,
        ignore_index=True
    )
    display(final_repair_results)
else:
    print("No metric files found. We will use the verified results table.")

In [ ]:
classification_dir = project_root / "results" / "classification"

rf_summary = pd.read_csv(
    classification_dir / "classification_method_summary.csv"
)

lr_summary = pd.read_csv(
    classification_dir / "logistic_regression_method_summary.csv"
)

majority_baseline = pd.read_csv(
    classification_dir / "majority_baseline.csv"
)

print("=== RANDOM FOREST SUMMARY ===")
display(rf_summary)

print("=== LOGISTIC REGRESSION SUMMARY ===")
display(lr_summary)

print("=== MAJORITY CLASS BASELINE ===")
display(majority_baseline)

In [ ]:
operational_dir = project_root / "results" / "operational"

for seed in [42, 123, 2026]:
    print(f"\n{'='*15} SEED {seed} {'='*15}")

    files_to_check = [
        f"llm_operational_metrics_seed_{seed}.csv",
        f"hitl_review_metrics_seed_{seed}.csv",
    ]

    for filename in files_to_check:
        path = operational_dir / filename

        print(f"\nFile: {filename}")

        if path.exists() and path.stat().st_size > 2:
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            display(df)
        else:
            print("Missing or empty")

In [ ]:
operational_dir = project_root / "results" / "operational"

rows = []

for seed in [42, 123, 2026]:
    df = pd.read_csv(
        operational_dir / f"llm_operational_metrics_seed_{seed}.csv"
    )

    review_path = (
        operational_dir / f"hitl_review_metrics_seed_{seed}.csv"
    )

    review_time = None
    approved = None
    rejected = None

    if review_path.exists() and review_path.stat().st_size > 2:
        review = pd.read_csv(review_path).iloc[0]
        review_time = review["review_time_seconds"]
        approved = review["approved"]
        rejected = review["rejected"]

    rows.append({
        "seed": seed,
        "batches": len(df),
        "runtime_seconds": df["runtime_seconds"].sum(),
        "prompt_tokens": df["prompt_tokens"].sum(),
        "completion_tokens": df["completion_tokens"].sum(),
        "total_tokens": (
            df["prompt_tokens"].sum()
            + df["completion_tokens"].sum()
        ),
        "approved": approved,
        "rejected": rejected,
        "review_time_seconds": review_time
    })

operational_final = pd.DataFrame(rows)

display(operational_final.round(2))

print("\nMean LLM runtime:")
print(operational_final["runtime_seconds"].mean().round(2), "seconds")

print("\nMean HITL review time (documented seeds only):")
print(
    operational_final["review_time_seconds"]
    .mean()
    .round(2),
    "seconds"
)

In [ ]:
seed42_path = (
    operational_dir / "llm_operational_metrics_seed_42.csv"
)

seed42_df = pd.read_csv(seed42_path)

print("Rows:", len(seed42_df))
print("Runtime sum:", seed42_df["runtime_seconds"].sum())
print("Runtime mean:", seed42_df["runtime_seconds"].mean())
print("Minimum:", seed42_df["runtime_seconds"].min())
print("Maximum:", seed42_df["runtime_seconds"].max())

print("\nSaved operational files:")
for path in sorted(operational_dir.glob("*seed_42*")):
    print(path.name, "-", path.stat().st_size, "bytes")